# 🚛 Optimización de Carga de Camiones — v38

**Algoritmo de packing 3D con restricciones logísticas operativas.**

### Criterios de prioridad (orden estricto):
0. **PRIORIDAD MÁXIMA:** Agotar todos los embalajes de un proveedor antes de pasar al siguiente
1. Agrupamiento homogéneo por tipo de embalaje (dentro de cada proveedor)
2. Orden por volumen total descendente (dentro de cada proveedor)
3. Orientación uniforme obligatoria por tipo de embalaje
4. Maximización: altura → ancho → largo
5. Continuidad espacial ordenada
5bis. Uniformización de niveles dentro de una misma área longitudinal
6. Llenado de espacio residual con embalajes pendientes del mismo proveedor
7. Múltiples camiones automáticos

### Regla de apilamiento entre tipos distintos:
- **Regla general:** dos tipos de embalaje distintos no se apilan uno encima del otro.
- **Excepción de apilamiento mixto:** si dos tipos de embalaje pertenecen al **mismo proveedor (DUNS)** y tienen **idénticas dimensiones de piso (Largo × Ancho)**, pueden apilarse en la misma columna vertical. El segundo tipo ocupa la altura residual de la columna iniciada por el primero, respetando su propia restricción de niveles y el límite de `CAMION_ALTO`.
- Para todos los demás casos, la regla general se garantiza por arquitectura: cada tipo ocupa un tramo longitudinal exclusivo, de piso a techo.

### Reglas de datos:
- Cada Part Number tiene un único tipo de embalaje por proveedor
- Un PN puede quedar repartido entre más de un camión si sus embalajes no caben en uno solo

### Archivos de input requeridos:
Los 7 archivos `.xlsx` deben estar en la misma carpeta que este notebook:
`tabla_embalajes.xlsx`, `tabla_demanda.xlsx`, `tabla_apilamiento.xlsx`, `tabla_camion.xlsx`, `tabla_proveedores.xlsx`, `tabla_embalajes_paletizados.xlsx`, `tabla_ventanas_colecta.xlsx`

`tabla_embalajes_paletizados.xlsx` es opcional: si no existe o está vacía, el script omite la sustitución y procede con los embalajes originales.

### Archivos de output generados:
- `resultado_optimizacion.xlsx` — 11 hojas: 6 de output (Detalle, Resumen por Camión, Resumen por PN, Superficie Mínima por PN, Matriz Distancias, Estadísticas + Análisis de Ocupación) y 5 de input
- `dashboard_ocupacion_camiones.html` — dashboard interactivo autocontenido con 11 paneles

### Tablas de salida principales:
- **Detalle por PN y Camión** — una fila por Part Number por camión
- **Resumen por Camión** — derivado 100% desde el detalle, garantizando coherencia
- **Resumen por Part Number** — superficie y camiones necesarios por PN
- **Superficie Mínima por PN** — cálculo independiente de superficie mínima teórica por PN

---
## 📋 Revisiones

| Versión | Descripción de cambios |
|---------|------------------------|
| **v1**  | Estructura inicial del algoritmo de optimización de carga 3D con criterios de agrupamiento homogéneo, orientación uniforme y múltiples camiones. |
| **v2**  | Incorporación de prioridad por proveedor (DUNS): se cargan todos los embalajes de un proveedor antes de pasar al siguiente. Datos de ejemplo ampliados a 10 proveedores, 50 Part Numbers y 30 embalajes. Salida en formato tabla. |
| **v3**  | Granularidad de la tabla de salida bajada a nivel de Part Number por camión. Cada fila indica cantidad de piezas del PN, embalajes, orientación y volumen en m³. Hoja adicional de PNs repartidos entre camiones. |
| **v4**  | Regla explícita: un Part Number tiene un único embalaje por proveedor. Llenado de espacio residual del camión con embalajes pendientes del mismo proveedor. Consistencia total entre las tres tablas de salida derivando resumen y PNs repartidos desde la tabla de detalle. |
| **v5**  | Nueva columna **Orientación** (L/T) en la tabla de detalle, indicando si el embalaje se carga longitudinal o transversal respecto al ancho del camión. |
| **v6**  | Regla de no apilamiento entre tipos distintos de embalaje formalizada, documentada y verificada. Validación automática en celda de verificaciones. |
| **v7**  | Incorporación de columnas **Superficie ocupada (m²)**, **Sup. total camión (m²)** y **Superficie ocupada %** en la hoja Resumen por Camión, calculadas como área de piso de nivel 1 de cada bloque. |
| **v8**  | Eliminación de columnas no solicitadas (Largo usado, Largo residual) del resumen. Eliminación de la hoja PNs Repartidos. |
| **v9**  | Nueva hoja 3 **Resumen por Part Number** con superficie mínima necesaria, longitud requerida y camiones necesarios por PN, calculados usando la orientación óptima del algoritmo. |
| **v10** | Integración del cálculo de superficie mínima por Part Number directamente en el script (eliminando dependencia de script externo). Nueva hoja 4 **Superficie mínima por PN**. |
| **v11** | Reemplazo del cálculo de camiones necesarios en la hoja 4 por **Longitud Requerida (cm)** y nuevo cálculo de camiones como `Longitud Requerida / Largo Camión`. |
| **v12** | Incorporación de `tabla_proveedores.xlsx` como nuevo input. Cálculo de la **matriz de distancias lineales entre proveedores** (Haversine) y exportación como hoja 5 **Matriz Distancias Proveedores**. |
| **v13** | Renumeración secuencial de todas las secciones del script (eliminando sub-secciones 3b, 3c y BIS). Actualización del título principal a v13 y registro de versión en el nombre del archivo. |
| **v14** | Corrección del algoritmo de optimización: (1) no mezcla de DUNS por tipo de embalaje en un mismo camión, (2) máximo 3 proveedores por camión, (3) zonas longitudinales continuas por proveedor sin intercalado, (4) el espacio residual de un proveedor solo puede llenarse con carga de otro proveedor en el extremo final y solo si el tipo de embalaje no está ya en el camión, (5) prioridad de proveedores por proximidad geográfica usando la matriz de distancias Haversine. |
| **v15** | Corrección del error `distancia_entre not defined` agregando la función en la celda de matriz de distancias. Ampliación de descripciones en todas las celdas del notebook. Nueva hoja 6 **Estadísticas de Ocupación** con histogramas de volumen (m³) y superficie (m²) cargados por camión. |
| **v16** | Eliminación de la regla que impedía que un mismo tipo de embalaje coexistiera en el mismo camión para distintos proveedores. Los embalajes están etiquetados individualmente y son identificables, por lo que la restricción no es necesaria. Se mantiene el criterio de no intercalado longitudinal entre zonas de distintos proveedores. |
| **v17** | Corrección de `MAX_PROVS_POR_CAMION` no definido: se agrega como constante en la celda de parámetros del camión. Renumeración de secciones: estadísticas pasa a ser sección 13 y exportar a Excel pasa a ser sección 14. |
| **v18** | Correcciones del análisis exhaustivo: (1) SyntaxError en celda de exportación por dos print en la misma línea, (2) variable `c` protegida en algoritmo para caso pendiente==0, (3) variable `largo_res` sin uso eliminada, (4) Validación 4 actualizada para reflejar que la mezcla de DUNS por embalaje está permitida desde v16 — ahora valida no intercalado longitudinal, (5) Sección de Importaciones actualizada con matplotlib e io, (6) comentario incorrecto 'celda 10' corregido a 'celda 14', (7) renumeración de secciones: estadísticas pasa a 12 y exportar a 13. |
| **v19** | Agregado análisis del bottom 25% de camiones con menor ocupación en la hoja 6 del archivo de salida. Para cada camión del cuartil inferior se calculan dinámicamente las causas (restricción de apilamiento, residuo transversal, residuo longitudinal, límite de proveedores) y una recomendación específica para mejorar la ocupación. Los comentarios se generan a partir de los datos reales del algoritmo y se presentan en una tabla formateada debajo de los histogramas. |
| **v20** | Las 5 tablas de input se incluyen ahora como hojas adicionales del archivo de salida (tabla_embalajes_input, tabla_demanda_input, tabla_apilamiento_input, tabla_camion_input, tabla_proveedores_input). Coloración de pestañas: gris para hojas de input, verde para hojas de output. |
| **v21** | Generación de dashboard HTML interactivo (`dashboard_ocupacion_camiones.html`) como output adicional. El dashboard replica el diseño del dashboard de referencia e integra todos los datos del análisis en 10 paneles con navegación por tabs: Resumen, Superficie, Volumen, Restricciones, Proveedores, Distancias, Detalle Carga, Sup. Mínima PN, Resumen PN e Inputs. |
| **v22** | Incorporación del campo `analisis` en el objeto de datos del dashboard HTML. Contiene la lista de camiones del bottom 25% de ocupación volumétrica con sus causas y recomendaciones calculadas dinámicamente, permitiendo que el panel **Análisis Ocupación** del dashboard se genere con datos reales. |
| **v23** | Tres criterios adicionales de análisis en `_analizar_camion`: (5) orientación subóptima (`ORIENT.`): detecta cuando la orientación contraria del embalaje hubiera cargado más unidades dado el largo final real del camión; (6) embalaje con baja cobertura estructural del camión (`EMBALAJE ÚNICO`): cuando un proveedor usa un solo tipo de embalaje que no aprovecha bien ni el ancho ni la altura del camión; (7) demanda insuficiente (`DEMANDA`): cuando el largo usado es menor al 60% y toda la demanda del proveedor ya fue cargada. Incorporación de fuentes Overpass (light 300, semibold 600, bold 700) embebidas en base64 en el dashboard HTML generado, reemplazando IBM Plex. |
| **v24** | Correcciones de auditoría: (C1) se reintegran `gc`, `BASE` y `textColor` al `dashboard_template.html` — habían quedado fuera del template al extraerlo, causando que todos los gráficos y tablas del dashboard se renderizaran vacíos; (M1) se reemplaza `IBM Plex Mono` por `Overpass` dentro de `BASE` para consistencia de fuente en los ticks de Chart.js; (M2) se corrige la indentación del Criterio 5 (`ORIENT.`) en `_analizar_camion` — estaba dentro del bloque `if n_provs == 3` en lugar de al mismo nivel que los demás criterios; (m1) se elimina la variable `niv_usado` que se asignaba pero nunca se consumía. |
| **v25** | Cuatro ajustes: (1) `pd.read_excel()` ahora incluye `engine='openpyxl'` como parámetro explícito en los cinco llamados de la celda de carga; (2) tabla de revisiones reordenada de v1 a v25 en orden cronológico ascendente; (3) análisis de ocupación cambiado de bottom 25% a bottom 50% (cuantil 0.50), actualizado tanto en la celda de análisis como en el título de la hoja Excel; (4) `prov_stats` en la celda del dashboard ahora incluye `dir`, `lat` y `lon` para cada proveedor, y la tabla de proveedores del panel Inputs muestra Dirección, Latitud y Longitud. |
| **v26** | Revisión completa de descripciones de celdas: (0) presentación general ampliada con lista de inputs, outputs y tablas de salida; (3) referencia a "Sección de Importaciones" corregida a "Importaciones"; (7) descripción ahora menciona `MAX_PROVS_POR_CAMION`; (13) descripción ahora menciona renombrado de columnas y cálculo de volúmenes; (15) corrección del orden de proveedores: el vecino más cercano reordena completamente, el volumen solo fija el punto de partida; (21) Tabla 3 corregida — cálculo independiente, no derivado de celda 4; (23) "cuatro" corregido a "cinco" validaciones y se agrega mención de estadísticas finales; (25) descripción ampliada con los 7 criterios de `_analizar_camion` y el análisis del bottom 50%; (27) Hoja 6 ahora menciona la tabla de análisis de causas; (28) panel Análisis Ocupación agregado a la lista del dashboard con sus 7 tipos de causa; (29 y 30) comentarios inline "bottom 25%" corregidos a "bottom 50%". |
| **v27** | Nueva excepción a la regla de apilamiento entre tipos distintos: dos tipos de embalaje del mismo proveedor (DUNS) con idénticas dimensiones de piso (Largo × Ancho) pueden apilarse en la misma columna vertical. Cambios de implementación: (1) `niveles_max()` acepta parámetro `alto_usado_cm` para calcular niveles sobre altura parcialmente ocupada; (2) nueva función `bloques_compatibles()` identifica bloques existentes con los que es posible el apilamiento mixto; (3) nueva función `calcular_bloque_sobre_existente()` calcula el bloque apilado sobre uno existente sin avanzar el largo del camión; (4) `registrar_bloque()` maneja el flag `solapado=True` para no incrementar `largo_usado`; (5) el algoritmo principal intenta el apilamiento mixto antes de abrir un nuevo tramo longitudinal; (6) Validación 5 actualizada para no reportar como error las superposiciones del tipo permitido; (7) todas las descripciones markdown actualizadas. |
| **v28** | Nueva sección 3 — Sustitución de embalajes paletizados: antes de calcular cantidades de embalajes, el script lee `tabla_embalajes_paletizados.xlsx` y sobreescribe en `df_embalajes` el embalaje, las dimensiones (Largo, Ancho, Alto) y los PNs por embalaje para los pares DUNS+PartNumber que tienen embalaje paletizado. Los pares sin entrada en esa tabla siguen el flujo estándar sin modificación. La restricción de apilamiento del embalaje paletizado se aplica automáticamente vía el merge con `df_apilamiento`; la restricción del embalaje original queda inactiva para ese par. Las secciones 3–14 se renumeraron a 4–15. Se actualizaron referencias cruzadas a números de celda en celdas 2, 12, 15, 23 y 27, y la lista de inputs en celdas 0 y 5. |
| **v29** | Nueva sección 15 — Cronograma de colectas: genera `df_cronograma` a partir de `tabla_ventanas_colecta.xlsx` y `df_resumen`. Para cada camión determina el día y horario de colecta por proveedor, considerando 1 hora de carga, 35 km/h de traslado y orden por ventana más temprana. Camiones consolidados se asignan al primer día con ventanas coincidentes para todos sus proveedores; si no existe ese día, se asignan por separado con marca ⚠️. Sección 15 (Dashboard HTML) renumerada a 16. Lista de inputs actualizada de 6 a 7 archivos en celdas 0 y 5. |
| **v30** | Corrección de cálculo de superficie ocupada: los bloques con `solapado=True` (apilamiento mixto introducido en v27) se excluyen de la fórmula de `Superficie ocupada %` y `Superficie ocupada (m²)` en la celda de construcción de tablas, ya que comparten huella de piso con el bloque base y no ocupan área adicional. Sin esta corrección la superficie podía superar el 100% (llegando al 344%) al contarse el mismo piso múltiples veces. Ajustes visuales en el dashboard template: título principal +2pt (25px), subtítulo del dashboard con el mismo color que el título, KPI labels +1pt (11px), card-titles +1pt (12px). Versión renumerada de v29 a v30. |
| **v31** | Robustez en columna ID Embalaje: se agrega fallback para casos donde el DUNS de `df_detalle` no tiene match en `df_proveedores` (Nombre Proveedor = NaN). En ese caso se usa el DUNS directamente como prefijo del ID Embalaje en lugar de producir un valor vacío. Sin este fallback, un DUNS sin match generaba `NaN_EMB-XX`. |
| **v32** | Tres ajustes en el dashboard template: (1) corrección del gráfico "Embalajes por camión" en el panel Resumen PN — el gráfico ahora respeta los filtros activos derivando los camiones relevantes desde `rpn_rows` filtrado en lugar de usar `D.labels` y `D.det_rows` directamente; (2) título principal h1 +3pt (25px → 28px); (3) color de la descripción del dashboard unificado con el color del título (`var(--text3)` → `var(--text)`). |
| **v33** | Corrección del campo `Orden` en `df_cronograma`: antes se asignaba según la posición del proveedor en `duns_list`, lo que producía órdenes invertidos cuando los proveedores caían en días distintos. Ahora se ordena por `Hora Inicio` dentro de cada grupo camión+día y se reasigna `Orden` con `cumcount()+1`, garantizando que el orden siempre refleja la secuencia cronológica real. Cambios en dashboard template: filtro por proveedor en panel Distancias, mapa de calor y gráfico de pares ampliados a todos los proveedores con scrollbar, barras del gráfico "Pares más cercanos" con altura dinámica y etiquetas completas. |
| **v34** | Corrección del cálculo de volumen ocupado por camión en `df_resumen`: el cálculo anterior sumaba `Volumen Ocupado (m³)` por fila de `df_detalle`, lo que producía doble conteo cuando múltiples PNs compartían el mismo bloque físico. Ahora el volumen se calcula directamente desde los bloques de `resultados` (`largo_ocupado × por_ancho × ancho_orientado × niveles × alto / 1_000_000`), garantizando que cada bloque físico se cuenta exactamente una vez. |
| **v36** | Corrección de bug en `optimizar_carga`: la llamada a `llenar_residual_mismo_prov` al final de cada ítem del plan estaba dentro del loop `for item in items_prov`, lo que la ejecutaba N veces (una por cada tipo de embalaje del proveedor). Esto causaba que los tipos de embalaje pendientes se colocaran en bloques superpuestos dentro del mismo camión, generando registros con posiciones idénticas y cantidades de embalajes incorrectas (ej: 197 en lugar de 72 para EMB-04 en camión 1). La corrección mueve esa llamada fuera del loop para que se ejecute exactamente una vez por proveedor, después de procesar todos sus tipos de embalaje. Corrección adicional en la generación del dashboard: el bloque `# ── Tablas de input originales` (que asigna `inp_embalajes`, `inp_demanda`, etc. a `_D`) estaba ubicado después de la serialización `_json.dumps(_D)` y la escritura del HTML, por lo que las 7 tablas de input siempre llegaban vacías al dashboard. El bloque se movió para ejecutarse antes de la serialización. Corrección de snapshots de input: `_df_paletizados_raw` y `_df_ventanas_raw` se capturaban después de transformaciones (renombrado de columnas, dropna, coerción de tipos); ambos snapshots se movieron a la línea inmediata posterior a su `pd.read_excel()` para reflejar el estado original del archivo. |
| **v37** | Corrección de bug en el mecanismo de apilamiento mixto (`bloques_compatibles` + `calcular_bloque_sobre_existente`): la altura vertical disponible para apilar un nuevo tipo de embalaje se calculaba solo sobre el último bloque en esa posición, sin acumular la altura de todos los bloques solapados anteriores. Esto permitía que el algoritmo asignara múltiples capas completas en el mismo espacio longitudinal, superando el límite físico del camión (ej: 197 embalajes EMB-04 en camión 1 contra un máximo real de 72). La corrección acumula la altura total de todos los bloques con el mismo `pos_inicio` y `DUNS` antes de calcular el espacio disponible, tanto en `bloques_compatibles` (para filtrar candidatos) como en `calcular_bloque_sobre_existente` (para calcular niveles reales disponibles). El parámetro `alto_acumulado` se agrega a la firma de `calcular_bloque_sobre_existente` y se calcula en el punto de llamada del algoritmo principal. |
| **v46** | Restricciones de consolidado entre proveedores: carga de `tabla_restricciones_consolidado.xlsx`, construcción del diccionario `_restr` y criterio en `camion_acepta_duns` que impide consolidar proveedores con restricción mutua o con restricción `Ninguno`. |
| **v38** | Corrección del bug residual en `bloques_compatibles`: la función no verificaba que el embalaje candidato fuera de tipo distinto al nuevo. Esto permitía que el mismo tipo de embalaje se apilara sobre su propio bloque con orientación diferente (físicamente imposible), asignando más unidades de las que caben en el espacio longitudinal disponible (ej: 120 en lugar de 72 para EMB-04 en camión 1 tras el fix de v37). Se agrega la condición `b['embalaje'] == embalaje_nuevo → continue` en el loop de la función, completando las tres condiciones del apilamiento mixto: (1) mismo DUNS, (2) mismo Largo × Ancho, (3) distinto código de embalaje. Se actualiza la firma de `bloques_compatibles` con el parámetro `embalaje_nuevo` y el punto de llamada en `optimizar_carga`. |



---
## 📦 Importaciones

| Librería | Import | Uso en el script |
|----------|--------|------------------|
| **pandas** | `import pandas as pd` | Lectura de archivos Excel, construcción y manipulación de DataFrames, merge de tablas, agrupaciones, exportación de resultados. Es la librería central del script. |
| **numpy** | `import numpy as np` | Operaciones vectorizadas sobre arrays: cálculo de `CEIL` para cantidad de embalajes, conversión de grados a radianes en Haversine, inicialización de la matriz de distancias. |
| **math** | `import math` | Funciones matemáticas puras (`floor`, `ceil`, `sqrt`, `arcsin`) usadas dentro de las funciones del algoritmo de optimización donde se opera fila por fila. |
| **collections.defaultdict** | `from collections import defaultdict` | Construcción de diccionarios con valor por defecto en las celdas de validación, para detectar mezcla de proveedores por bloque. Pertenece a la librería estándar de Python. |
| **IPython.display** | `from IPython.display import display` | Renderizado de DataFrames como tablas formateadas en el entorno Jupyter Notebook. Reemplaza a `print()` para mostrar tablas con formato visual. Requiere entorno Jupyter. |
| **openpyxl** | `from openpyxl.styles import Font, PatternFill, Alignment, Border, Side` / `from openpyxl.utils import get_column_letter` / `from openpyxl.drawing.image import Image as XLImage` | Motor de escritura `.xlsx` en `pd.ExcelWriter(..., engine='openpyxl')`. Los submódulos se importan explícitamente en las celdas 28 y 32 para aplicar estilos (fuente, relleno, alineación, bordes), calcular letras de columna y embeber la imagen de histogramas en la hoja de estadísticas. |
| **matplotlib** | `import matplotlib` / `import matplotlib.pyplot as plt` / `import matplotlib.ticker as ticker` | Generación de los histogramas de ocupación (volumen m³ y superficie m²) en la celda 13. Se configura con backend `Agg` para renderizado sin pantalla. |
| **io** | `import io` | Utilizado para crear un buffer en memoria (`BytesIO`) donde se guarda la imagen de los histogramas antes de embeberla en la hoja Excel de estadísticas. |
| **json** | `import json as _json` | Serialización del objeto de datos `_D` a formato JSON para inyectarlo en el archivo HTML del dashboard. Pertenece a la biblioteca estándar de Python. |
| **base64** | `import base64 as _b64` | Codificación en base64 de los archivos de fuentes `.woff2` para embeberlos directamente en el HTML del dashboard, eliminando la dependencia de archivos externos. Pertenece a la biblioteca estándar de Python. |
| **math** *(alias)* | `import math as _math` | Alias local usado en las celdas 28 y 31 para las funciones `floor` y `ceil` dentro de `_analizar_camion` y la preparación de datos del dashboard, sin reasignar el `import math` global de la celda 4. |
---


## 1. Importación de librerías

Carga todas las librerías necesarias para el script y configura las opciones de visualización de pandas. Todas las importaciones están documentadas en **Importaciones** al inicio del notebook.

In [225]:
import pandas as pd
import numpy as np
import math
from collections import defaultdict
from IPython.display import display
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 260)
pd.options.display.float_format = '{:.4f}'.format

## 2. Carga de tablas
> Los 7 archivos `.xlsx` deben estar en la misma carpeta que este notebook:
> `tabla_embalajes.xlsx`, `tabla_demanda.xlsx`, `tabla_apilamiento.xlsx`, `tabla_camion.xlsx`, `tabla_proveedores.xlsx`, `tabla_embalajes_paletizados.xlsx`, `tabla_ventanas_colecta.xlsx`
>
> `tabla_embalajes_paletizados.xlsx` es opcional: si no existe o está vacía, el script omite la sustitución y procede con los embalajes originales.

In [226]:
df_embalajes   = pd.read_excel('tabla_embalajes.xlsx',   engine='openpyxl')
df_demanda     = pd.read_excel('tabla_demanda.xlsx',     engine='openpyxl')
df_apilamiento = pd.read_excel('tabla_apilamiento.xlsx', engine='openpyxl')
df_camion      = pd.read_excel('tabla_camion.xlsx',      engine='openpyxl')
df_proveedores = pd.read_excel('tabla_proveedores.xlsx', engine='openpyxl')

# Camiones prearmados (opcional)
try:
    df_prearmados = pd.read_excel('tabla_camiones_prearmadaos.xlsx', engine='openpyxl')
    df_prearmados.columns = ['Camion', 'DUNS', 'Embalaje', 'CantidadEmbalajes']
    df_prearmados['DUNS'] = df_prearmados['DUNS'].astype(str)
    print(f'✅ tabla_camiones_prearmadaos.xlsx — {df_prearmados["Camion"].nunique()} camión/es prearmado/s')
except FileNotFoundError:
    df_prearmados = None
    print('ℹ️  tabla_camiones_prearmadaos.xlsx no encontrado')

# Restricciones de consolidado entre proveedores (opcional)
try:
    df_restricciones = pd.read_excel('tabla_restricciones_consolidado.xlsx', engine='openpyxl')
    df_restricciones.columns = ['DUNS', 'Nombre', 'NoConsolidaCon', 'GrupoConsolidacion']
    df_restricciones['DUNS'] = df_restricciones['DUNS'].astype(str)
    print(f'✅ tabla_restricciones_consolidado.xlsx — {len(df_restricciones)} restricción/es cargadas')
except FileNotFoundError:
    df_restricciones = None
    print('ℹ️  tabla_restricciones_consolidado.xlsx no encontrado — sin restricciones de consolidado')

# Guardar snapshots originales ANTES de cualquier procesamiento
_df_embalajes_raw   = df_embalajes.copy()
_df_demanda_raw     = df_demanda.copy()
_df_apilamiento_raw = df_apilamiento.copy()
_df_camion_raw          = df_camion.copy()
_df_proveedores_raw     = df_proveedores.copy()
_df_restricciones_raw   = df_restricciones.copy() if df_restricciones is not None else pd.DataFrame()

print('── Tabla Embalajes (primeras filas) ──'); display(df_embalajes.head())
print('── Tabla Demanda (primeras filas) ──');   display(df_demanda.head())
print('── Tabla Apilamiento (primeras filas) ──'); display(df_apilamiento.head())
print('── Tabla Camion ──'); display(df_camion)

✅ tabla_camiones_prearmadaos.xlsx — 2 camión/es prearmado/s
✅ tabla_restricciones_consolidado.xlsx — 20 restricción/es cargadas
── Tabla Embalajes (primeras filas) ──


,DUNS Proveedor,Part Number,Embalaje,Largo Embalaje (cm),Ancho Embalaje (cm),Alto Embalaje (cm),Cant. PNs por Embalaje
0,111000001,PN-026,EMB-01,60,40,30,10
1,111000001,PN-032,EMB-04,120,80,60,4
2,111000001,PN-044,EMB-11,75,55,42,8
3,111000001,PN-047,EMB-12,95,65,48,4
4,111000001,PN-006,EMB-01,60,40,30,10


── Tabla Demanda (primeras filas) ──


,DUNS Proveedor,Part Number,Cantidad de Part Number
0,111000001,PN-026,216
1,111000001,PN-032,481
2,111000001,PN-044,443
3,111000001,PN-047,447
4,111000001,PN-006,78


── Tabla Apilamiento (primeras filas) ──


,DUNS Proveedor,Empaque,Cant. Niveles de Apilamiento
0,111000001,EMB-04,2
1,111000001,EMB-11,5
2,111000001,EMB-12,2
3,222000002,EMB-02,2
4,222000002,EMB-05,4


── Tabla Camion ──


,Descripcion,Largo (cm),Ancho (cm),Alto (cm)
0,Camión Estándar,1500,270,280


## 3. Sustitución de embalajes paletizados

Antes de que cualquier celda posterior consuma `df_embalajes`, verifica si algún par
DUNS+PartNumber de la demanda tiene un embalaje paletizado definido en
`tabla_embalajes_paletizados.xlsx`. Para esos pares, sobreescribe en `df_embalajes`
el código de embalaje, las dimensiones (Largo, Ancho, Alto) y la cantidad de PNs por
embalaje por los valores del embalaje paletizado.

Los pares sin entrada en la tabla paletizada no se modifican y siguen el flujo estándar.

La restricción de apilamiento se aplica automáticamente: el merge posterior con
`df_apilamiento` (sección 7) resuelve el código paletizado (ej. `EMB-30_6`) y no el
embalaje original (`EMB-30`), por lo que solo rige la restricción del embalaje paletizado.
La restricción del embalaje original queda inactiva para ese par DUNS+PN.

**Input adicional requerido:** `tabla_embalajes_paletizados.xlsx` en la misma carpeta
que el notebook. Si el archivo no existe o no contiene filas, el script omite la
sustitución y procede con los embalajes originales sin errores.

In [227]:
# Leer tabla de embalajes paletizados
# header=0: los encabezados están en la primera fila del Excel
df_paletizados = pd.read_excel('tabla_embalajes_paletizados.xlsx',
                               engine='openpyxl', header=0)
_df_paletizados_raw = df_paletizados.rename(
    columns=lambda c: str(c).replace('\n', ' ')).copy()  # snapshot: limpia \n en encabezados
df_paletizados.columns = [
    'DUNS', 'PartNumber', '_pns_orig', '_emb_orig', '_cant_pallet',
    'Embalaje', 'PNsPorEmbalaje', '_cap_verif', '_emb_verif',
    'Largo', 'Ancho', 'Alto'
]
df_paletizados = df_paletizados[
    ['DUNS', 'PartNumber', 'Embalaje', 'PNsPorEmbalaje', 'Largo', 'Ancho', 'Alto']
].dropna(subset=['DUNS', 'PartNumber'])

# Normalizar DUNS a str en ambos DataFrames para evitar mismatch de tipo
# (Excel puede leer DUNS como int64 o str según el formato de celda)
df_paletizados['DUNS'] = df_paletizados['DUNS'].astype(str)
df_embalajes[df_embalajes.columns[0]] = df_embalajes[df_embalajes.columns[0]].astype(str)

# Construir lookup indexado por (DUNS, PartNumber)
_pal_lookup = df_paletizados.set_index(['DUNS', 'PartNumber'])

# Nombres de columna originales del Excel en df_embalajes (antes del renombrado de sección 7)
_cols_emb = df_embalajes.columns.tolist()
# Posiciones: [0]=DUNS (str), [1]=PartNumber, [2]=Embalaje, [3]=Largo, [4]=Ancho, [5]=Alto, [6]=PNsPorEmbalaje

_sustituidos = []
for idx, row in df_embalajes.iterrows():
    key = (row.iloc[0], row.iloc[1])  # (DUNS, PartNumber)
    if key in _pal_lookup.index:
        pal = _pal_lookup.loc[key]
        df_embalajes.at[idx, _cols_emb[2]] = str(pal['Embalaje'])        # código paletizado
        df_embalajes.at[idx, _cols_emb[3]] = int(pal['Largo'])           # 120 cm
        df_embalajes.at[idx, _cols_emb[4]] = int(pal['Ancho'])           # 80 cm
        df_embalajes.at[idx, _cols_emb[5]] = int(pal['Alto'])            # alto calculado
        df_embalajes.at[idx, _cols_emb[6]] = int(pal['PNsPorEmbalaje'])  # PNs por emb. paletizado
        _sustituidos.append((key[0], key[1], str(pal['Embalaje'])))

print(f'✅ Sustitución de embalajes paletizados: {len(_sustituidos)} par(es) sustituido(s)')
for _d, _p, _e in _sustituidos:
    print(f'   DUNS {_d}  {_p}  →  {_e}')
if not _sustituidos:
    print('   (ningún par DUNS+PN de la demanda tiene embalaje paletizado)')
del _pal_lookup, _cols_emb  # limpiar variables temporales del entorno global

✅ Sustitución de embalajes paletizados: 8 par(es) sustituido(s)
   DUNS 222000002  PN-053  →  EMB-30_6
   DUNS 666000006  PN-061  →  EMB-40_8
   DUNS 777000007  PN-064  →  EMB-50_6
   DUNS 101000010  PN-070  →  EMB-40_8
   DUNS 112000011  PN-071  →  EMB-30_6
   DUNS 112000011  PN-073  →  EMB-40_8
   DUNS 667000016  PN-086  →  EMB-30_6
   DUNS 889000018  PN-092  →  EMB-30_6


## 4. Parámetros del camión

Lee las dimensiones internas del camión desde `tabla_camion.xlsx` y las almacena como constantes globales (`CAMION_LARGO`, `CAMION_ANCHO`, `CAMION_ALTO`, `VOL_CAMION`). Estas constantes son utilizadas por todas las celdas posteriores del algoritmo para calcular capacidades, orientaciones, niveles de apilamiento y superficies.

También se define aquí `MAX_PROVS_POR_CAMION = 3`, la constante que limita la cantidad máxima de proveedores distintos que pueden cargar en un mismo camión, utilizada por la función `camion_acepta_duns()` del algoritmo.

In [228]:
CAMION_LARGO_FISICO  = int(df_camion['Largo (cm)'].iloc[0])
CAMION_LARGO_SEG     = int(df_camion['Largo  de seguridad(cm)'].iloc[0])  # margen de seguridad descontado del largo utilizable
CAMION_LARGO = CAMION_LARGO_FISICO - CAMION_LARGO_SEG  # largo realmente disponible para carga
CAMION_ANCHO = int(df_camion['Ancho (cm)'].iloc[0])
CAMION_ALTO  = int(df_camion['Alto (cm)'].iloc[0])
VOL_CAMION   = CAMION_LARGO * CAMION_ANCHO * CAMION_ALTO

print(f'Largo físico: {CAMION_LARGO_FISICO} cm | Largo de seguridad: {CAMION_LARGO_SEG} cm | Largo utilizable: {CAMION_LARGO} cm')
print(f'Ancho: {CAMION_ANCHO} cm | Alto: {CAMION_ALTO} cm')
print(f'Volumen total del camión: {VOL_CAMION:,} cm³  =  {VOL_CAMION/1_000_000:.4f} m³')
MAX_PROVS_POR_CAMION = 3  # Máximo de proveedores distintos permitidos por camión
print(f'Máximo proveedores por camión: {MAX_PROVS_POR_CAMION}')


Largo: 1500 cm | Ancho: 270 cm | Alto: 280 cm
Volumen total del camión: 113,400,000 cm³  =  113.4000 m³
Máximo proveedores por camión: 3


## 5. Cálculo de superficie mínima por Part Number

Calcula la superficie mínima de piso de camión necesaria para cargar la demanda de cada Part Number, de forma independiente al algoritmo de optimización. Para cada PN determina:
- La cantidad de embalajes necesarios (`CEIL(CantidadPN / PNsPorEmbalaje)`)
- Los niveles máximos de apilamiento permitidos (restricción física y de tabla)
- La orientación óptima que minimiza la longitud requerida
- La cantidad de columnas necesarias, la superficie por columna y la superficie mínima total
- La longitud requerida en cm y la cantidad de camiones necesarios según esa longitud

El resultado se exporta como **Hoja 4** del archivo de salida.

In [229]:
# Requiere que df_embalajes, df_demanda, df_apilamiento y parámetros del camión
# estén ya cargados (celdas 4, 6 y 8).
# Se ejecuta antes del renombrado de columnas de la celda 16,
# pero aún los nombres originales del Excel en cuanto a nombre de columna
# (el renombrado ocurre en la celda 16). Los valores de df_embalajes ya reflejan
# la sustitución paletizada aplicada en la celda 8.

_emb = df_embalajes.copy()
_dem = df_demanda.copy()
_api = df_apilamiento.copy()
_emb.columns = ['DUNS','PartNumber','Embalaje','Largo','Ancho','Alto','PNsPorEmbalaje','PrioridadCarga']
_dem.columns = ['DUNS','PartNumber','CantidadPN']
_api.columns = ['DUNS','Embalaje','NivelesApilamiento']

# Normalizar DUNS a str para consistencia con df_embalajes (coercionado en sección 3)
_dem['DUNS'] = _dem['DUNS'].astype(str)
_api['DUNS'] = _api['DUNS'].astype(str)

_df = pd.merge(_emb, _dem, on=['DUNS','PartNumber'], how='inner')
_df = pd.merge(_df, _api, on=['DUNS','Embalaje'], how='left')

_df['_CantEmb'] = np.ceil(_df['CantidadPN'] / _df['PNsPorEmbalaje']).astype(int)
_df['_NivMax']  = _df.apply(
    lambda r: min(math.floor(CAMION_ALTO / r['Alto']), int(r['NivelesApilamiento']))
              if pd.notna(r['NivelesApilamiento'])
              else math.floor(CAMION_ALTO / r['Alto']),
    axis=1
)
_df['_CantCol'] = np.ceil(_df['_CantEmb'] / _df['_NivMax']).astype(int)
_df['_SupCol']  = round(_df['Largo'] * _df['Ancho'] / 10_000, 6)
_df['_SupMin']  = round(_df['_CantCol'] * _df['_SupCol'], 4)
_sup_camion     = round(CAMION_LARGO * CAMION_ANCHO / 10_000, 4)

# ── Longitud requerida por PN ────────────────────────────────────────
# Para cada PN se simula un bloque hipotético exclusivo con todos sus embalajes.
# Se elige la orientación óptima (igual que el algoritmo principal) y se mide
# la longitud que ocuparía ese bloque en el camión.
def _longitud_requerida(cant_emb, largo, ancho, alto, nivel_rest):
    niv = math.floor(CAMION_ALTO / alto)
    if pd.notna(nivel_rest):
        niv = min(niv, int(nivel_rest))
    # Evaluar las dos orientaciones y elegir la que minimiza longitud
    mejor_lo, mejor_long = None, float('inf')
    for lo, ao in [(largo, ancho), (ancho, largo)]:
        qa = math.floor(CAMION_ANCHO / ao)
        if qa == 0 or niv == 0:
            continue
        cant_col = math.ceil(cant_emb / (niv * qa))
        longitud = cant_col * lo
        if longitud < mejor_long:
            mejor_long = longitud
            mejor_lo   = lo
    return mejor_long if mejor_lo is not None else None

_df['_LongReq'] = _df.apply(
    lambda r: _longitud_requerida(
        r['_CantEmb'], r['Largo'], r['Ancho'], r['Alto'], r['NivelesApilamiento']
    ), axis=1
)
_df['_CamNec'] = round(_df['_LongReq'] / CAMION_LARGO, 1)

df_sup_minima = _df[[
    'DUNS','PartNumber','CantidadPN','Embalaje',
    '_CantEmb','Largo','Ancho','Alto',
    '_CantCol','_SupCol','_SupMin','_LongReq','_CamNec'
]].copy()
df_sup_minima.columns = [
    'DUNS Proveedor','Part Number','Cantidad de Demanda (PN)','Embalaje',
    'Cant. Unidades de Embalaje','Largo Embalaje (cm)','Ancho Embalaje (cm)','Alto Embalaje (cm)',
    'Cant. Columnas de Embalaje','Sup. por Columna (m²)',
    'Sup. Mínima Necesaria (m²)','Longitud Requerida (cm)','Camiones Necesarios'
]

print(f'✅ Superficie mínima calculada — {len(df_sup_minima)} Part Numbers')
display(df_sup_minima.head())

✅ Superficie mínima calculada — 10 Part Numbers


,DUNS Proveedor,Part Number,Cantidad de Demanda (PN),Embalaje,Cant. Unidades de Embalaje,Largo Embalaje (cm),Ancho Embalaje (cm),Alto Embalaje (cm),Cant. Columnas de Embalaje,Sup. por Columna (m²),Sup. Mínima Necesaria (m²),Longitud Requerida (cm),Camiones Necesarios
0,111000001,PN-026,216,EMB-01,22,60,40,30,3,0.2400,0.7200,40,0.0000
1,111000001,PN-032,481,EMB-04,121,120,80,60,61,0.9600,58.5600,2480,1.7000
2,111000001,PN-044,443,EMB-11,56,75,55,42,12,0.4125,4.9500,220,0.1000
3,111000001,PN-047,447,EMB-12,112,95,65,48,56,0.6175,34.5800,1330,0.9000
4,111000001,PN-006,78,EMB-01,8,60,40,30,1,0.2400,0.2400,40,0.0000


## 6. Matriz de distancias entre proveedores

Calcula la distancia lineal en kilómetros entre cada par de proveedores usando la **fórmula de Haversine**, que tiene en cuenta la curvatura de la Tierra. Las coordenadas de cada proveedor provienen de `tabla_proveedores.xlsx`.

El resultado es una matriz simétrica de `N×N` proveedores donde cada celda indica la distancia en km. La diagonal vale 0 (distancia de un proveedor consigo mismo).

Esta matriz cumple dos funciones en el script:
- Se usa en el **algoritmo de optimización** para priorizar proveedores geográficamente cercanos al compartir un camión
- Se exporta como **Hoja 5** del archivo de salida

También se define aquí la función auxiliar `distancia_entre(duns_a, duns_b)` que es invocada por el algoritmo.

In [230]:
df_proveedores.columns = ['DUNS', 'Nombre', 'Direccion', 'Latitud', 'Longitud']

def _haversine(lat1, lon1, lat2, lon2):
    R = 6371
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    return round(2 * R * np.arcsin(np.sqrt(a)), 1)

df_matriz = pd.DataFrame(
    index=df_proveedores['DUNS'],
    columns=df_proveedores['DUNS'],
    dtype=float
)
for i, ri in df_proveedores.iterrows():
    for j, rj in df_proveedores.iterrows():
        df_matriz.loc[ri['DUNS'], rj['DUNS']] = (
            0.0 if i == j
            else _haversine(ri['Latitud'], ri['Longitud'], rj['Latitud'], rj['Longitud'])
        )

print(f'✅ Matriz de distancias calculada: {len(df_proveedores)}×{len(df_proveedores)}')
print(f'   Distancia máxima: {df_matriz.values.max():.1f} km')
print(f'   Distancia mínima (sin diagonal): {df_matriz.replace(0, np.nan).min().min():.1f} km')
display(df_matriz)

def distancia_entre(duns_a, duns_b):
    """Devuelve la distancia en km entre dos proveedores según la matriz Haversine."""
    try:
        return float(df_matriz.loc[duns_a, duns_b])
    except:
        return float('inf')


✅ Matriz de distancias calculada: 20×20
   Distancia máxima: 950.3 km
   Distancia mínima (sin diagonal): 0.2 km


DUNS,111000001,222000002,333000003,444000004,555000005,112000011,223000012,334000013,666000006,777000007,888000008,999000009,101000010,445000014,556000015,667000016,778000017,889000018,990000019,100100020
DUNS,,,,,,,,,,,,,,,,,,,,
111000001,0.0000,0.8000,8.5000,4.1000,1.5000,3.6000,1.0000,8.4000,86.9000,83.4000,89.3000,86.1000,82.6000,76.3000,137.6000,48.6000,290.8000,769.0000,335.2000,376.7000
222000002,0.8000,0.0000,8.8000,4.9000,2.2000,4.2000,0.2000,8.5000,86.1000,82.6000,89.5000,86.3000,82.1000,76.6000,136.9000,47.9000,290.0000,769.2000,335.2000,376.9000
333000003,8.5000,8.8000,0.0000,7.1000,7.8000,10.3000,8.9000,2.8000,90.4000,87.6000,96.7000,93.4000,79.2000,83.3000,139.6000,53.2000,295.7000,760.5000,327.2000,368.2000
444000004,4.1000,4.9000,7.1000,0.0000,2.6000,3.4000,5.1000,8.4000,90.9000,87.5000,89.5000,86.2000,84.8000,76.1000,141.5000,52.7000,294.9000,767.1000,334.3000,374.8000
555000005,1.5000,2.2000,7.8000,2.6000,0.0000,2.9000,2.5000,8.2000,88.3000,84.9000,89.4000,86.2000,83.4000,76.3000,139.0000,50.1000,292.2000,768.3000,334.8000,376.0000
112000011,3.6000,4.2000,10.3000,3.4000,2.9000,0.0000,4.4000,11.0000,89.6000,85.9000,86.5000,83.3000,86.2000,73.4000,140.7000,51.1000,292.8000,770.5000,337.5000,378.2000
223000012,1.0000,0.2000,8.9000,5.1000,2.5000,4.4000,0.0000,8.5000,85.8000,82.4000,89.5000,86.4000,81.9000,76.7000,136.6000,47.6000,289.8000,769.3000,335.1000,376.9000
334000013,8.4000,8.5000,2.8000,8.4000,8.2000,11.0000,8.5000,0.0000,87.7000,85.0000,97.6000,94.3000,76.9000,84.4000,136.9000,50.7000,293.2000,760.8000,326.7000,368.5000
666000006,86.9000,86.1000,90.4000,90.9000,88.3000,89.6000,85.8000,87.7000,0.0000,8.0000,134.7000,134.1000,69.0000,130.1000,54.1000,39.0000,207.3000,798.9000,347.1000,410.0000


## 7. Preparación y consolidación de datos

Renombra las columnas de `df_embalajes`, `df_demanda` y `df_apilamiento` con nombres estandarizados que utilizan todas las celdas posteriores del algoritmo. En este punto `df_embalajes` ya contiene las sustituciones aplicadas en la sección 3 (embalajes paletizados reemplazados con sus dimensiones y código de pallet). Luego integra las tres tablas en un único DataFrame de trabajo mediante operaciones de merge. Calcula la cantidad de embalajes necesarios para cubrir la demanda de cada Part Number (`CantidadEmbalajes = CEIL(CantidadPN / PNsPorEmbalaje)`), incorpora las restricciones de apilamiento por tipo de embalaje, y calcula `VolEmbalaje_cm3` y `VolEmbalaje_m3` para cada tipo de embalaje.

También valida que cada Part Number tenga un único tipo de embalaje asignado por proveedor, condición necesaria para el correcto funcionamiento del algoritmo.

In [231]:
df_embalajes.columns   = ['DUNS', 'PartNumber', 'Embalaje', 'Largo', 'Ancho', 'Alto', 'PNsPorEmbalaje', 'PrioridadCarga']
df_demanda.columns     = ['DUNS', 'PartNumber', 'CantidadPN']
df_apilamiento.columns = ['DUNS', 'Embalaje', 'NivelesApilamiento']

# Normalizar DUNS a str en df_demanda y df_apilamiento para que el merge
# sea consistente con df_embalajes (coercionado a str en la sección 3)
df_demanda['DUNS']     = df_demanda['DUNS'].astype(str)
df_apilamiento['DUNS'] = df_apilamiento['DUNS'].astype(str)

df = pd.merge(df_embalajes, df_demanda, on=['DUNS', 'PartNumber'], how='inner')
df['CantidadEmbalajes'] = np.ceil(df['CantidadPN'] / df['PNsPorEmbalaje']).astype(int)
df = pd.merge(df, df_apilamiento, on=['DUNS', 'Embalaje'], how='left')
df['VolEmbalaje_cm3'] = df['Largo'] * df['Ancho'] * df['Alto']
df['VolEmbalaje_m3']  = df['VolEmbalaje_cm3'] / 1_000_000

# Validar que cada PN tiene un único embalaje por proveedor
multi = df.groupby(['DUNS','PartNumber'])['Embalaje'].nunique()
if (multi > 1).any():
    print('⚠️  Advertencia: hay PNs con más de un embalaje por proveedor:')
    display(multi[multi > 1])
else:
    print('✅ Validación OK: cada PN tiene un único embalaje por proveedor')

print(f'\nTotal de Part Numbers consolidados: {len(df)}')
display(df[['DUNS','PartNumber','Embalaje','Largo','Ancho','Alto',
            'CantidadPN','PNsPorEmbalaje','CantidadEmbalajes',
            'NivelesApilamiento','VolEmbalaje_m3']].head(12))

✅ Validación OK: cada PN tiene un único embalaje por proveedor

Total de Part Numbers consolidados: 10


,DUNS,PartNumber,Embalaje,Largo,Ancho,Alto,CantidadPN,PNsPorEmbalaje,CantidadEmbalajes,NivelesApilamiento,VolEmbalaje_m3
0,111000001,PN-026,EMB-01,60,40,30,216,10,22,NaN,0.0720
1,111000001,PN-032,EMB-04,120,80,60,481,4,121,2.0000,0.5760
2,111000001,PN-044,EMB-11,75,55,42,443,8,56,5.0000,0.1732
3,111000001,PN-047,EMB-12,95,65,48,447,4,112,2.0000,0.2964
4,111000001,PN-006,EMB-01,60,40,30,78,10,8,NaN,0.0720
5,999000009,PN-008,EMB-02,80,50,40,74,6,13,NaN,0.1600
6,999000009,PN-067,EMB-39,144,94,71,53,5,11,NaN,0.9611
7,999000009,PN-068,EMB-43,117,71,55,235,4,59,NaN,0.4569
8,667000016,PN-087,EMB-48,123,75,60,3500,10,350,NaN,0.5535
9,667000016,PN-088,EMB-49,143,91,68,5430,5,1086,NaN,0.8849


## 8. Ordenamiento del plan de carga

Construye el plan de carga ordenado según las dos reglas de prioridad principales:

**Regla 0 — Prioridad por proveedor:** todos los embalajes de un proveedor se cargan antes de pasar al siguiente. El orden de proveedores se determina en dos pasos: primero se ordena por volumen total descendente para establecer el proveedor inicial, y luego se aplica el **algoritmo del vecino más cercano** sobre la matriz de distancias Haversine, que reordena completamente la lista priorizando la proximidad geográfica. El orden por volumen solo fija el punto de partida; la secuencia final la define la proximidad.

**Regla 2 — Prioridad por volumen dentro del proveedor:** dentro de cada proveedor, los tipos de embalaje se ordenan por volumen total descendente.

In [232]:
df_emb = df.groupby(
    ['DUNS', 'Embalaje', 'Largo', 'Ancho', 'Alto', 'NivelesApilamiento'],
    dropna=False
).agg(CantidadEmbalajes=('CantidadEmbalajes','sum')).reset_index()

df_emb['VolUnitario'] = df_emb['Largo'] * df_emb['Ancho'] * df_emb['Alto']
df_emb['VolTotal']    = df_emb['VolUnitario'] * df_emb['CantidadEmbalajes']

# Altura utilizada = Alto × min(NivelesApilamiento, floor(CAMION_ALTO/Alto))
df_emb['AltUsada'] = df_emb.apply(
    lambda r: r['Alto'] * (min(int(r['NivelesApilamiento']), math.floor(CAMION_ALTO / r['Alto']))
                           if pd.notna(r['NivelesApilamiento'])
                           else r['Alto'] * math.floor(CAMION_ALTO / r['Alto'])),
    axis=1
)

vol_prov = df_emb.groupby('DUNS')['VolTotal'].sum().rename('VolProv')
df_emb   = df_emb.merge(vol_prov, on='DUNS')

# Prioridad de carga por embalaje: mínimo entre todos sus PartNumbers
# (mínimo = valor más prioritario). NaN cuando ningún PN tiene prioridad asignada.
_prio_emb = df[df['PrioridadCarga'].notna()].groupby(
    ['DUNS','Embalaje'])['PrioridadCarga'].min().rename('_PrioMin')
df_emb = df_emb.merge(_prio_emb, on=['DUNS','Embalaje'], how='left')
# Reemplazar NaN por inf para que los embalajes sin prioridad queden al final del grupo
df_emb['_PrioSort'] = df_emb['_PrioMin'].fillna(float('inf'))

df_emb   = df_emb.sort_values(
    ['VolProv','DUNS','_PrioSort','AltUsada'], ascending=[False, True, True, False]
).reset_index(drop=True)

# Eliminar columnas auxiliares de prioridad para no contaminar el resto del flujo
df_emb.drop(columns=['_PrioMin','_PrioSort'], inplace=True)

df_pn = df[['DUNS','PartNumber','Embalaje','CantidadPN','CantidadEmbalajes',
'PNsPorEmbalaje','Largo','Ancho','Alto','NivelesApilamiento',
            'VolEmbalaje_cm3','VolEmbalaje_m3']].copy()
df_pn = df_pn.sort_values(['DUNS','Embalaje']).reset_index(drop=True)

print('Plan de carga (primeras filas):')
display(df_emb[['DUNS','Embalaje','Largo','Ancho','Alto',
                'NivelesApilamiento','CantidadEmbalajes','VolTotal','VolProv']].head(12))

Plan de carga (primeras filas):


,DUNS,Embalaje,Largo,Ancho,Alto,NivelesApilamiento,CantidadEmbalajes,VolTotal,VolProv
0,667000016,EMB-49,143,91,68,NaN,1086,960984024,1154709024
1,667000016,EMB-48,123,75,60,NaN,350,193725000,1154709024
2,111000001,EMB-01,60,40,30,NaN,30,2160000,114754800
3,111000001,EMB-11,75,55,42,5.0000,56,9702000,114754800
4,111000001,EMB-04,120,80,60,2.0000,121,69696000,114754800
5,111000001,EMB-12,95,65,48,2.0000,112,33196800,114754800
6,999000009,EMB-43,117,71,55,NaN,59,26956215,39607831
7,999000009,EMB-39,144,94,71,NaN,11,10571616,39607831
8,999000009,EMB-02,80,50,40,NaN,13,2080000,39607831


## 9. Funciones auxiliares

Define todas las funciones que utiliza el algoritmo principal. Se organizan en tres grupos:

**Funciones de cálculo físico:** `niveles_max()` determina el máximo de niveles de apilamiento respetando tanto el alto del camión como la restricción de la tabla; `mejor_orient()` evalúa las dos orientaciones horizontales posibles y elige la que maximiza la cantidad cargada; `uniformizar()` redistribuye embalajes para minimizar diferencias de altura entre columnas (criterio 5bis); `calcular_bloque()` integra todo lo anterior para determinar cuántos embalajes entran en un espacio disponible.

**Funciones de registro:** `desagregar_pns()` distribuye los embalajes de un bloque entre los Part Numbers que comparten ese tipo de embalaje; `registrar_bloque()` agrega el bloque al camión y actualiza el estado — si el bloque tiene el flag `solapado=True`, no avanza `largo_usado` porque comparte huella con un bloque existente compatible.

**Funciones de control de restricciones:** `duns_en_camion()` consulta el estado actual del camión; `camion_acepta_duns()` verifica el límite de 3 proveedores; `proveedor_finalizo_zona()` detecta si otro proveedor ya cargó después de la zona del proveedor activo (evita intercalado); `llenar_residual_mismo_prov()` ocupa el espacio residual con embalajes del mismo proveedor; `llenar_residual_otro_prov()` ocupa el residual final con un proveedor distinto; `ordenar_por_proximidad()` reordena los proveedores usando el vecino más cercano; `bloques_compatibles()` identifica bloques existentes del mismo DUNS y mismo L×A sobre los cuales puede apilarse un nuevo tipo; `calcular_bloque_sobre_existente()` calcula cuántos embalajes del nuevo tipo caben en la altura residual de una columna ya iniciada.

In [233]:
def niveles_max(alto_emb, nivel_rest, alto_usado_cm=0):
    """
    Calcula el máximo de niveles apilables para un embalaje.

    Args:
        alto_emb (int): Alto del embalaje en cm.
        nivel_rest: Restricción de la tabla de apilamiento (puede ser NaN).
        alto_usado_cm (int): Altura en cm ya ocupada por otro tipo de embalaje
            en la misma columna (caso de apilamiento mixto compatible). Por
            defecto 0 (columna libre desde el piso).

    Returns:
        int: Número máximo de niveles permitidos.
    """
    espacio_disponible = CAMION_ALTO - alto_usado_cm
    if espacio_disponible <= 0: return 0
    f = math.floor(espacio_disponible / alto_emb)
    if pd.notna(nivel_rest): return min(f, int(nivel_rest))
    return f

def mejor_orient(L,A,H,largo_disp,nivel_rest):
    mejor,mejor_qty=None,-1
    for (l,a) in [(L,A),(A,L)]:
        niv=niveles_max(H,nivel_rest); qa=math.floor(CAMION_ANCHO/a); ql=math.floor(largo_disp/l)
        if niv==0 or qa==0 or ql==0: continue
        qty=niv*qa*ql
        if qty>mejor_qty: mejor_qty=qty; mejor=dict(lo=l,ao=a,ho=H,niv=niv,qa=qa,ql=ql)
    return mejor

def uniformizar(qty,qa,niv_max_val):
    if qa==0: return 0,0
    ql=math.ceil(qty/(qa*niv_max_val)); niv=min(math.ceil(qty/(qa*ql)),niv_max_val)
    return ql,niv

def calcular_bloque(row,largo_disp,pendiente,orient_fija=None):
    L,A,H=row['Largo'],row['Ancho'],row['Alto']; nr=row['NivelesApilamiento']
    if orient_fija: lo,ao=orient_fija['lo'],orient_fija['ao']
    else:
        o=mejor_orient(L,A,H,largo_disp,nr)
        if o is None: return None
        lo,ao=o['lo'],o['ao']
    niv=niveles_max(H,nr); qa=math.floor(CAMION_ANCHO/ao)
    if niv==0 or qa==0: return None
    ql_max=math.floor(largo_disp/lo); ql_nec=math.ceil(pendiente/(niv*qa))
    ql_real=min(ql_max,ql_nec)
    if ql_real==0: return None
    qty=min(pendiente,ql_real*qa*niv)
    ql_u,niv_u=uniformizar(qty,qa,niv); ql_u=min(ql_u,ql_max)
    if ql_u==0: return None
    return dict(lo=lo,ao=ao,ho=H,niv=niv_u,qa=qa,ql=ql_u,qty=qty,largo_ocu=ql_u*lo)

def desagregar_pns(pns_emb,estado_pn,cant_emb_bloque):
    filas_pn=[]; emb_restantes=cant_emb_bloque
    for key in pns_emb:
        if emb_restantes<=0: break
        st=estado_pn[key]
        if st['emb_pend']<=0: continue
        emb_este_pn=min(st['emb_pend'],emb_restantes)
        pns_completos=(emb_este_pn-1)*st['pns_por_emb'] if emb_este_pn>1 else 0
        pns_ultimo=min(st['pn_pend']-pns_completos,st['pns_por_emb'])
        pns_este_pn=min(pns_completos+pns_ultimo,st['pn_pend'])
        filas_pn.append({'part_number':key[1],'cant_pn':pns_este_pn,'cant_emb':emb_este_pn,
                         'vol_m3':round(emb_este_pn*st['vol_emb_m3'],6),'pns_por_emb':st['pns_por_emb']})
        st['emb_pend']-=emb_este_pn; st['pn_pend']-=pns_este_pn; emb_restantes-=emb_este_pn
    return filas_pn,cant_emb_bloque-emb_restantes

def registrar_bloque(c, item, bloque, estado_pn):
    """
    Registra un bloque de embalajes en el camión y actualiza el estado.

    Si el bloque tiene el flag 'solapado=True', se apila sobre un bloque
    existente compatible (mismo DUNS, mismo L×A) sin avanzar largo_usado.
    En caso contrario, ocupa un nuevo tramo longitudinal.

    Args:
        c (dict): Camión en construcción.
        item (dict): Ítem del plan de carga.
        bloque (dict): Resultado de calcular_bloque() o calcular_bloque_sobre_existente().
        estado_pn (dict): Estado de pendientes por Part Number.
    """
    filas_pn, emb_real = desagregar_pns(item['pns_emb'], estado_pn, bloque['qty'])
    row = item['row']
    solapado = bloque.get('solapado', False)
    pos_ini  = bloque['pos_inicio'] if solapado else c['largo_usado']
    c['bloques'].append(dict(
        duns=row['DUNS'], embalaje=row['Embalaje'],
        largo_emb=row['Largo'], ancho_emb=row['Ancho'], alto_emb=row['Alto'],
        orientacion=f"{bloque['lo']} x {bloque['ao']} x {bloque['ho']}",
        orient_tipo='L' if bloque['ao'] == max(row['Largo'], row['Ancho']) else 'T',
        qty_emb_total=emb_real, niveles=bloque['niv'],
        por_ancho=bloque['qa'], por_largo=bloque['ql'],
        largo_ocupado=bloque['largo_ocu'],
        pos_inicio=pos_ini, pos_fin=pos_ini + bloque['largo_ocu'],
        detalle_pn=filas_pn,
        solapado=solapado  # True si comparte huella con otro tipo compatible
    ))
    # Solo avanzar largo_usado si el bloque ocupa un nuevo tramo longitudinal
    if not solapado:
        c['largo_usado'] += bloque['largo_ocu']
    item['pendiente'] -= emb_real
    if item['orient_fija'] is None:
        item['orient_fija'] = dict(lo=bloque['lo'], ao=bloque['ao'])

def bloques_compatibles(c, duns, largo_emb, ancho_emb, embalaje_nuevo):
    """
    Devuelve los bloques del camión con los que un nuevo embalaje puede
    compartir huella de piso (apilamiento mixto permitido).

    Condición de compatibilidad: mismo DUNS, misma orientación de piso en el
    camión (lo y ao idénticos entre el bloque existente y el nuevo embalaje),
    y espacio vertical aún disponible en la columna.

    La comparación de orientación es direccional (no como set): el embalaje
    nuevo debe poder adoptar exactamente el mismo lo y ao que el bloque
    existente. Comparar como set {L,A} == {L,A} era incorrecto porque igualaba
    embalajes con el mismo par de dimensiones pero distintas orientaciones en
    el camión (ej: lo=260,ao=100 vs lo=100,ao=260), generando bloques solapados
    con orientaciones incompatibles que producían embalajes "flotantes".

    Args:
        c (dict): Camión actual con sus bloques.
        duns: DUNS del proveedor del embalaje a colocar.
        largo_emb (int): Largo nominal del embalaje (sin orientar).
        ancho_emb (int): Ancho nominal del embalaje (sin orientar).
        embalaje_nuevo (str): Código del embalaje a colocar. Solo bloques de
            tipo distinto son candidatos válidos: el solapado mixto aplica
            únicamente entre tipos de embalaje diferentes con igual huella de piso.

    Returns:
        list[dict]: Lista de bloques existentes compatibles.
    """
    # Las dos orientaciones posibles del nuevo embalaje en el camión
    orientaciones_posibles = {(largo_emb, ancho_emb), (ancho_emb, largo_emb)}
    compatibles = []
    for b in c['bloques']:
        if b['duns'] != duns:
            continue
        # El solapado mixto aplica solo entre tipos DISTINTOS de embalaje.
        if b['embalaje'] == embalaje_nuevo:
            continue
        # La orientación de piso del bloque existente debe coincidir exactamente
        # con una de las orientaciones posibles del nuevo embalaje.
        # Se lee lo/ao directamente del campo 'orientacion' del bloque base,
        # que es la fuente de verdad sobre cómo está ubicado en el camión.
        _dims_base = [int(x.strip()) for x in b['orientacion'].split('x')]
        lo_base, ao_base = _dims_base[0], _dims_base[1]
        if (lo_base, ao_base) not in orientaciones_posibles:
            continue
        # Calcular altura total acumulada en esta posición sumando TODOS los bloques
        # (solapados y no solapados) que comparten el mismo pos_inicio y DUNS.
        alto_ya_usado = sum(
            x['niveles'] * x['alto_emb']
            for x in c['bloques']
            if x['pos_inicio'] == b['pos_inicio'] and x['duns'] == b['duns']
        )
        if CAMION_ALTO - alto_ya_usado > 0:
            compatibles.append(b)
    return compatibles


def calcular_bloque_sobre_existente(row, bloque_base, pendiente, alto_acumulado, orient_fija=None):
    """
    Calcula cuántos embalajes del nuevo tipo caben sobre un bloque existente
    compatible (mismo DUNS, mismo L×A), compartiendo su rango longitudinal.

    El nuevo tipo ocupa la altura residual de la columna ya iniciada.
    El rango longitudinal (largo_ocu, pos_inicio, pos_fin) se hereda del bloque_base.

    Args:
        row (pd.Series): Fila del plan de carga del nuevo embalaje.
        bloque_base (dict): Bloque existente sobre el cual se apilará.
        pendiente (int): Embalajes pendientes del nuevo tipo.
        orient_fija (dict | None): Orientación fija si ya fue determinada.

    Returns:
        dict | None: Bloque calculado con largo_ocu heredado, o None si no cabe nada.

    Criterio:
        Se usa la misma orientación que el bloque_base para garantizar que las
        dimensiones de piso coincidan. Los niveles se calculan sobre el espacio
        vertical restante (CAMION_ALTO - altura_ya_ocupada).
    """
    alto_usado_cm = alto_acumulado  # altura total acumulada en esta posición (todos los bloques solapados)
    # Usar la misma orientación de piso que el bloque base, leyendo directamente
    # el campo 'orientacion' que ya tiene los valores lo/ao correctamente orientados.
    # Reconstruir lo/ao desde orient_tipo y largo_emb/ancho_emb era incorrecto
    # cuando Largo < Ancho, porque la asignación dependía del valor relativo.
    _dims_base = [int(x.strip()) for x in bloque_base['orientacion'].split('x')]
    lo = _dims_base[0]
    ao = _dims_base[1]
    niv = niveles_max(row['Alto'], row['NivelesApilamiento'], alto_usado_cm)
    qa  = math.floor(CAMION_ANCHO / ao)
    if niv == 0 or qa == 0: return None
    # El rango longitudinal es el del bloque_base
    ql = bloque_base['por_largo']
    qty = min(pendiente, ql * qa * niv)
    if qty == 0: return None
    ql_u, niv_u = uniformizar(qty, qa, niv)
    ql_u = min(ql_u, ql)  # no puede superar el largo del bloque base
    if ql_u == 0: return None
    return dict(
        lo=lo, ao=ao, ho=row['Alto'],
        niv=niv_u, qa=qa, ql=ql_u,
        qty=qty,
        largo_ocu=bloque_base['largo_ocupado'],  # hereda el largo del bloque base
        pos_inicio=bloque_base['pos_inicio'],     # hereda la posición del bloque base
        solapado=True  # indica que no avanza largo_usado del camión
    )


def duns_en_camion(c): return set(b['duns'] for b in c['bloques'])


def camion_acepta_duns(c, duns):
    """
    Verifica si el camión puede aceptar carga del proveedor indicado.

    Criterios (en orden, con prioridad decreciente):
    1. Si el proveedor ya tiene carga en el camión → siempre acepta.
    2. Si se supera MAX_PROVS_POR_CAMION → rechaza.
    3. Si la columna 'No Consolida Con' indica que el candidato o algún
       proveedor ya presente no puede consolidar con el otro → rechaza.
       Este criterio tiene prioridad sobre el criterio 4.
    4. Si la columna 'Grupo de consolidación' indica que el candidato y
       algún proveedor ya presente pertenecen a grupos distintos → rechaza.
       Proveedores del mismo grupo solo pueden consolidar entre sí.

    Args:
        c (dict): Camión activo con su lista de bloques.
        duns (str): DUNS del proveedor candidato a cargar.

    Returns:
        bool: True si el camión acepta la carga, False si la rechaza.
    """
    presentes = duns_en_camion(c)
    if duns in presentes: return True
    if len(presentes) >= MAX_PROVS_POR_CAMION: return False
    # Verificar restricciones de consolidado
    if _restr:
        # ¿El candidato tiene restricción 'Ninguno'?
        if _restr.get(duns) == 'Ninguno' and presentes:
            return False
        # ¿Algún proveedor ya presente tiene restricción 'Ninguno'?
        for p in presentes:
            if _restr.get(p) == 'Ninguno':
                return False
        # ¿El candidato lista algún proveedor presente entre sus exclusiones?
        excl_cand = _restr.get(duns, set())
        if isinstance(excl_cand, set) and excl_cand & presentes:
            return False
        # ¿Algún proveedor presente lista al candidato entre sus exclusiones?
        for p in presentes:
            excl_p = _restr.get(p, set())
            if isinstance(excl_p, set) and duns in excl_p:
                return False
    # Verificar restricciones de grupo de consolidación
    # (aplica solo si el candidato no fue ya rechazado por NoConsolidaCon)
    gc_cand = _grupos_consolidacion.get(duns)
    if gc_cand is not None:
        for p in presentes:
            if _grupos_consolidacion.get(p) != gc_cand:
                return False
    for p in presentes:
        gc_p = _grupos_consolidacion.get(p)
        if gc_p is not None and _grupos_consolidacion.get(duns) != gc_p:
            return False
    return True

def proveedor_finalizo_zona(c, duns):
    ultimo_fin_duns  = max((b['pos_fin'] for b in c['bloques'] if b['duns']==duns),  default=0)
    ultimo_fin_otros = max((b['pos_fin'] for b in c['bloques'] if b['duns']!=duns),  default=0)
    return ultimo_fin_otros > ultimo_fin_duns

def llenar_residual_mismo_prov(c, items_prov, estado_pn):
    for item in items_prov:
        if item['pendiente']<=0: continue
        # Intentar primero apilamiento sobre bloque compatible del mismo proveedor
        compat = bloques_compatibles(
            c, item['row']['DUNS'],
            item['row']['Largo'], item['row']['Ancho'],
            item['row']['Embalaje']
        )
        if compat:
            _alto_acum = sum(
                x['niveles'] * x['alto_emb']
                for x in c['bloques']
                if x['pos_inicio'] == compat[-1]['pos_inicio'] and x['duns'] == compat[-1]['duns']
            )
            bloque_s = calcular_bloque_sobre_existente(
                pd.Series(item['row']), compat[-1], item['pendiente'], _alto_acum, item['orient_fija']
            )
            if bloque_s is not None:
                registrar_bloque(c, item, bloque_s, estado_pn)
                continue
        # Si no hay apilamiento posible, intentar tramo longitudinal nuevo
        largo_disp=CAMION_LARGO-c['largo_usado']
        if largo_disp<=0: break
        bloque=calcular_bloque(pd.Series(item['row']),largo_disp,item['pendiente'],item['orient_fija'])
        if bloque is None: continue
        registrar_bloque(c,item,bloque,estado_pn)

def llenar_residual_otro_prov(c, item_otro, estado_pn):
    """
    Rellena el espacio residual final del camión con un proveedor distinto.
    Condiciones:
    - El camión acepta un proveedor más (máximo 3 proveedores).
    - Solo actúa sobre el extremo final (largo_usado actual).
    Un mismo tipo de embalaje puede estar en el camión de más de un proveedor
    porque los embalajes están etiquetados e identificables individualmente.
    """
    if not camion_acepta_duns(c, item_otro['row']['DUNS']): return
    # Intentar primero apilamiento sobre bloque compatible del mismo proveedor
    compat = bloques_compatibles(
        c, item_otro['row']['DUNS'],
        item_otro['row']['Largo'], item_otro['row']['Ancho'],
        item_otro['row']['Embalaje']
    )
    if compat:
        _alto_acum = sum(
            x['niveles'] * x['alto_emb']
            for x in c['bloques']
            if x['pos_inicio'] == compat[-1]['pos_inicio'] and x['duns'] == compat[-1]['duns']
        )
        bloque_s = calcular_bloque_sobre_existente(
            pd.Series(item_otro['row']), compat[-1], item_otro['pendiente'],
            _alto_acum, item_otro['orient_fija']
        )
        if bloque_s is not None:
            registrar_bloque(c, item_otro, bloque_s, estado_pn)
            return
    # Si no hay apilamiento posible, intentar tramo longitudinal nuevo
    largo_disp=CAMION_LARGO-c['largo_usado']
    if largo_disp<=0: return
    bloque=calcular_bloque(pd.Series(item_otro['row']),largo_disp,item_otro['pendiente'],item_otro['orient_fija'])
    if bloque is None: return
    registrar_bloque(c,item_otro,bloque,estado_pn)

def ordenar_por_proximidad(proveedores_orden):
    if len(proveedores_orden)<=1: return proveedores_orden
    restantes=list(proveedores_orden[1:]); ordenados=[proveedores_orden[0]]
    while restantes:
        ultimo=ordenados[-1]
        mas_cercano=min(restantes,key=lambda d: distancia_entre(ultimo,d))
        ordenados.append(mas_cercano); restantes.remove(mas_cercano)
    return ordenados

## 10. Algoritmo principal de optimización

Ejecuta el algoritmo de packing 3D con todas las restricciones logísticas. Itera sobre los proveedores en el orden definido en la celda anterior y para cada uno carga sus embalajes respetando esta jerarquía:

1. **Prioridad máxima — proveedor completo primero:** no se pasa al siguiente proveedor hasta agotar todos sus embalajes
2. **Máximo 3 proveedores por camión:** controlado por `camion_acepta_duns()`
3. **Zonas longitudinales continuas:** cada proveedor ocupa un tramo longitudinal exclusivo sin intercalado
4. **Residual solo en el extremo final:** al terminar un proveedor, el espacio residual puede ser ocupado por el proveedor pendiente más cercano geográficamente. Un mismo tipo de embalaje puede estar en el camión de distintos proveedores porque los embalajes están etiquetados individualmente
4bis. **Apilamiento mixto compatible:** antes de abrir un nuevo tramo longitudinal, el algoritmo verifica si el embalaje puede apilarse sobre un bloque existente del mismo DUNS con igual L×A. Si hay espacio vertical disponible, usa esa posición sin consumir más largo del camión
5. **Orientación uniforme:** cada tipo de embalaje mantiene la misma orientación en todos los camiones
6. **Uniformización de niveles (criterio 5bis):** redistribuye embalajes para minimizar diferencias de altura

In [234]:
# Construir diccionarios de restricciones de consolidado
# _restr[duns] = 'Ninguno'           → no consolida con nadie
# _restr[duns] = {duns_a, duns_b}    → no consolida con esos DUNS específicos
# _restr[duns] ausente               → sin restricción por exclusión explícita
#
# _grupos_consolidacion[duns] = 'X'  → el proveedor pertenece al grupo X
# _grupos_consolidacion[duns] ausente → sin restricción de grupo
# Prioridad: _restr tiene precedencia sobre _grupos_consolidacion.
# Si un proveedor está restringido por _restr, esa restricción aplica
# aunque pertenezca a un grupo que en principio lo habilitaría.
_restr = {}
_grupos_consolidacion = {}
if df_restricciones is not None and not df_restricciones.empty:
    for _, _rr in df_restricciones.iterrows():
        _d  = str(_rr['DUNS'])
        _nc = str(_rr['NoConsolidaCon']).strip() if pd.notna(_rr['NoConsolidaCon']) else ''
        if _nc and _nc != 'nan':
            if _nc.lower() == 'ninguno':
                _restr[_d] = 'Ninguno'
            else:
                _restr[_d] = {s.strip() for s in _nc.split(',') if s.strip()}
        _gc = str(_rr['GrupoConsolidacion']).strip() if pd.notna(_rr['GrupoConsolidacion']) else ''
        if _gc and _gc != 'nan':
            _grupos_consolidacion[_d] = _gc

def optimizar_carga(df_emb, df_pn, df_prearmados=None):
    camiones=[]; camion_id=1
    estado_pn={}
    for _,r in df_pn.iterrows():
        estado_pn[(r['DUNS'],r['PartNumber'])]={
            'embalaje':r['Embalaje'],'emb_pend':r['CantidadEmbalajes'],
            'pns_por_emb':r['PNsPorEmbalaje'],'pn_total':r['CantidadPN'],
            'pn_pend':r['CantidadPN'],'vol_emb_cm3':r['VolEmbalaje_cm3'],'vol_emb_m3':r['VolEmbalaje_m3'],
        }
    plan=[]
    for _,row in df_emb.iterrows():
        plan.append({'row':row.to_dict(),'pendiente':int(row['CantidadEmbalajes']),
                     'orient_fija':None,
                     'pns_emb':[k for k,v in estado_pn.items() if k[0]==row['DUNS'] and v['embalaje']==row['Embalaje']]})

    def nuevo_camion():
        nonlocal camion_id
        c=dict(id=camion_id,bloques=[],largo_usado=0); camiones.append(c); camion_id+=1; return c

    # ── Fase 0: resolver camiones prearmados ─────────────────────────
    # Para cada camión prearmado factible, ejecuta el loop estándar del
    # algoritmo acotando la demanda de cada ítem a la cantidad prearmada.
    # Reutiliza calcular_bloque y registrar_bloque sin modificarlos.
    _ids_prearmados = []  # IDs reales de camiones generados por prearmado
    if df_prearmados is not None and not df_prearmados.empty:
        _grupos_raw = {cn: grp.to_dict('records')
                       for cn, grp in df_prearmados.groupby('Camion')}

        # ── Verificar viabilidad y calcular volumen estimado de cada plantilla ──
        # Un prearmado es viable si todos sus ítems caben físicamente en el camión
        # actual (dimensiones + niveles de apilamiento) independientemente de la
        # demanda disponible. Esto evita armar camiones imposibles cuando cambian
        # las dimensiones del camión o de los embalajes.
        def _vol_prearmado(items):
            """
            Estima el volumen total (cm³) que ocuparía el prearmado en un camión
            vacío y verifica que todos sus ítems son físicamente cargables.

            Args:
                items (list[dict]): Ítems del prearmado con claves DUNS, Embalaje,
                    CantidadEmbalajes.

            Returns:
                float: Volumen estimado en cm³ si el prearmado es viable.
                None: Si algún ítem no cabe físicamente en el camión actual.

            Criterio:
                Simula calcular_bloque para cada ítem con largo_disp = CAMION_LARGO
                (camión vacío) y la cantidad exacta del prearmado. Si algún bloque
                devuelve None, el ítem no es cargable y el prearmado es inviable.
                El volumen se calcula como qty × Largo × Ancho × Alto para cada ítem.
            """
            vol_total = 0
            for _it in items:
                _duns = str(_it['DUNS'])
                _cant = int(_it['CantidadEmbalajes'])
                _ip = next((p for p in plan
                            if p['row']['DUNS'] == _duns
                            and p['row']['Embalaje'] == _it['Embalaje']), None)
                if _ip is None:
                    return None  # embalaje no existe en el plan actual
                _b = calcular_bloque(pd.Series(_ip['row']), CAMION_LARGO, _cant, None)
                if _b is None:
                    return None  # ítem no cabe en el camión actual
                # Volumen: cantidad de embalajes × dimensiones unitarias
                r = _ip['row']
                vol_total += _b['qty'] * r['Largo'] * r['Ancho'] * r['Alto']
            return vol_total

        # Filtrar plantillas viables y ordenarlas por volumen descendente
        _grupos_viables = []
        for _cn, _items in _grupos_raw.items():
            _vol = _vol_prearmado(_items)
            if _vol is not None:
                _grupos_viables.append((_cn, _items, _vol))
            else:
                print(f'⚠️  Prearmado {_cn} descartado: uno o más embalajes no '
                      f'son cargables con las dimensiones actuales del camión.')

        _grupos_viables.sort(key=lambda x: x[2], reverse=True)  # mayor vol primero
        _grupos = {_cn: _items for _cn, _items, _ in _grupos_viables}

        if _grupos_viables:
            print('📋 Orden de prearmados por volumen estimado:')
            for _cn, _items, _vol in _grupos_viables:
                print(f'   Prearmado {_cn}: {round(_vol/1e6, 3)} m³')

        _hubo = True
        while _hubo:
            _hubo = False
            for _cn in list(_grupos.keys()):
                _items = _grupos[_cn]
                # Verificar disponibilidad de stock
                _ok = all(
                    sum(v['emb_pend'] for k, v in estado_pn.items()
                        if k[0] == str(_it['DUNS']) and v['embalaje'] == _it['Embalaje'])
                    >= int(_it['CantidadEmbalajes'])
                    for _it in _items
                )
                if not _ok:
                    continue
                # Armar el camión usando el loop estándar del algoritmo
                c = nuevo_camion()
                _ids_prearmados.append(c['id'])  # registrar ID real generado
                for _it in _items:
                    _duns = str(_it['DUNS'])
                    _cant = int(_it['CantidadEmbalajes'])
                    _ip = next((p for p in plan
                               if p['row']['DUNS'] == _duns
                               and p['row']['Embalaje'] == _it['Embalaje']), None)
                    if _ip is None: continue
                    # Guardar pendiente original y acotarlo al prearmado
                    _pend_orig = _ip['pendiente']
                    _ip['pendiente'] = _cant
                    _row_s = pd.Series(_ip['row'])
                    _pendiente_item = _cant
                    while _pendiente_item > 0:
                        _largo_disp = CAMION_LARGO - c['largo_usado']
                        if _largo_disp <= 0: break
                        _b = calcular_bloque(_row_s, _largo_disp,
                                             _pendiente_item, _ip['orient_fija'])
                        if _b is None: break
                        registrar_bloque(c, _ip, _b, estado_pn)
                        _pendiente_item = _ip['pendiente']
                    # Restaurar pendiente real (registrar_bloque ya descontó _cant)
                    _ip['pendiente'] = _pend_orig - (_cant - _ip['pendiente'])
                # El camión prearmado queda cerrado exactamente con lo que indica
                # la plantilla. No se agrega carga adicional aunque haya espacio libre.
                _hubo = True

    # ── Fase 1: loop principal sobre proveedores ──────────────────────
    proveedores_orden=list(dict.fromkeys(p['row']['DUNS'] for p in plan if p['pendiente']>0))
    proveedores_orden=ordenar_por_proximidad(proveedores_orden)

    for duns_activo in proveedores_orden:
        items_prov=[p for p in plan if p['row']['DUNS']==duns_activo and p['pendiente']>0]

        for item in items_prov:
            row_series=pd.Series(item['row'])
            while item['pendiente']>0:
                if not camiones: nuevo_camion()
                # No usar camiones prearmados para carga de la Fase 1
                if camiones[-1]['id'] in _ids_prearmados: nuevo_camion()
                c=camiones[-1]
                # No intercalar: si el proveedor ya cerró su zona en este camión, abrir nuevo
                if c['bloques'] and duns_activo in duns_en_camion(c) and proveedor_finalizo_zona(c,duns_activo):
                    nuevo_camion(); continue
                if not camion_acepta_duns(c,duns_activo):
                    nuevo_camion(); continue
                # ── Intentar apilamiento mixto sobre bloque compatible ──
                # Si hay un bloque del mismo DUNS con igual L×A y espacio
                # vertical libre, apilar sobre él antes de abrir nuevo tramo.
                compat = bloques_compatibles(
                    c, item['row']['DUNS'],
                    item['row']['Largo'], item['row']['Ancho'],
                    item['row']['Embalaje']
                )
                if compat:
                    _alto_acum = sum(
                        x['niveles'] * x['alto_emb']
                        for x in c['bloques']
                        if x['pos_inicio'] == compat[-1]['pos_inicio'] and x['duns'] == compat[-1]['duns']
                    )
                    bloque_s = calcular_bloque_sobre_existente(
                        row_series, compat[-1], item['pendiente'], _alto_acum, item['orient_fija']
                    )
                    if bloque_s is not None:
                        registrar_bloque(c, item, bloque_s, estado_pn)
                        continue
                # ── Tramo longitudinal nuevo (comportamiento estándar) ──
                largo_disp=CAMION_LARGO-c['largo_usado']
                if largo_disp<=0:
                    llenar_residual_mismo_prov(c,items_prov,estado_pn)
                    nuevo_camion(); continue
                # Al iniciar un camión vacío, resetear orient_fija para que el
                # algoritmo elija la orientación óptima en ese camión, sin heredar
                # la orientación del camión anterior.
                if not c['bloques']:
                    item['orient_fija'] = None
                bloque=calcular_bloque(row_series,largo_disp,item['pendiente'],item['orient_fija'])
                if bloque is None:
                    llenar_residual_mismo_prov(c,items_prov,estado_pn)
                    nuevo_camion(); continue
                registrar_bloque(c,item,bloque,estado_pn)
        # Llenar residual una sola vez tras procesar todos los ítems del proveedor
        # (movido fuera del for-item para evitar que se ejecute N veces — una por
        # cada tipo de embalaje del proveedor — causando bloques superpuestos, v36)
        if camiones and camiones[-1]['id'] not in _ids_prearmados:
            llenar_residual_mismo_prov(camiones[-1],items_prov,estado_pn)

        # Al terminar todos los embalajes del proveedor activo,
        # intentar llenar el residual final con proveedores cercanos pendientes
        if camiones and camiones[-1]['id'] not in _ids_prearmados:
            c=camiones[-1]
            if CAMION_LARGO-c['largo_usado']>0:
                candidatos=[p for p in plan
                            if p['row']['DUNS']!=duns_activo and p['pendiente']>0
                            and camion_acepta_duns(c,p['row']['DUNS'])]
                candidatos.sort(key=lambda p: distancia_entre(duns_activo,p['row']['DUNS']))
                for cand in candidatos:
                    if cand['pendiente']<=0: continue
                    llenar_residual_otro_prov(c,cand,estado_pn)

    # Exponer IDs prearmados al scope global para serialización
    global _ids_prearmados_global
    _ids_prearmados_global = _ids_prearmados
    return camiones

resultados = optimizar_carga(df_emb, df_pn, df_prearmados=df_prearmados)
print(f'✅ Algoritmo finalizado — Camiones utilizados: {len(resultados)}')

📋 Orden de prearmados por volumen estimado:
   Prearmado 1: 69.042 m³
   Prearmado 2: 69.042 m³
✅ Algoritmo finalizado — Camiones utilizados: 21


## 11. Construcción de tablas de salida

Construye las tres tablas principales del archivo de salida a partir de los resultados del algoritmo. Las tablas 2 y 3 se derivan completamente desde la Tabla 1, garantizando coherencia total.

**Tabla 1 — Detalle por PN y Camión:** una fila por Part Number por camión. Incluye DUNS, Part Number, cantidad de piezas, embalaje, orientación, niveles, distribución espacial y volumen en m³. Un mismo PN puede aparecer en múltiples filas si quedó repartido entre camiones.

**Tabla 2 — Resumen por Camión:** una fila por camión con totales de PNs, embalajes, superficie ocupada (m² y %), volumen ocupado (m³ y %) y proveedores que cargaron en ese camión.

**Tabla 3 — Resumen por Part Number:** una fila por PN con DUNS, Part Number, cantidad de piezas, embalaje, PNs por embalaje, cantidad de embalajes, superficie ocupada mínima (m²) y camiones necesarios. El cálculo de superficie y camiones se realiza de forma independiente a la celda 5 — ambas celdas aplican la misma lógica pero sobre distintos DataFrames de origen (la celda 5 opera con los datos ya modificados por la sustitución paletizada de la celda 8).

In [235]:
# ── Tabla 1: Detalle por PN y Camión ────────────────────────────────
filas_detalle = []
for camion in resultados:
    for bloque_idx, b in enumerate(camion['bloques']):
        for pn_det in b['detalle_pn']:
            filas_detalle.append({
                'Camión N°':              camion['id'],
                'DUNS Proveedor':         b['duns'],
                'Part Number':            pn_det['part_number'],
                'Cant. Piezas (PN)':      pn_det['cant_pn'],
                'Embalaje':               b['embalaje'],
                'Cant. Embalajes':        pn_det['cant_emb'],
                'Orientación L×A×H (cm)': b['orientacion'],
                'Orientación':            b['orient_tipo'],
                'Niveles (Alto)':         b['niveles'],
                'Unid. por Ancho':        b['por_ancho'],
                'Unid. por Largo':        b['por_largo'],
                'Largo Ocupado (cm)':     b['largo_ocupado'],
                'Pos. Inicio (cm)':       b['pos_inicio'],
                'Pos. Fin (cm)':          b['pos_fin'],
                'Volumen Ocupado (m³)':   pn_det['vol_m3'],
                'Bloque Idx':             bloque_idx,
                'Solapado':               b.get('solapado', False),
            })

df_detalle = pd.DataFrame(filas_detalle)

# ── Agregar Nombre Proveedor e ID Embalaje al detalle ───────────
# Nombre Proveedor: lookup desde df_proveedores por DUNS
_duns_nombre = df_proveedores.set_index(
    df_proveedores['DUNS'].astype(str))['Nombre'].to_dict()
df_detalle['Nombre Proveedor'] = df_detalle['DUNS Proveedor'].astype(str).map(_duns_nombre)

# ID Embalaje: primera_palabra_del_nombre + '_' + código_de_embalaje
# Si el nombre tiene una sola palabra se usa completo; si tiene más, solo la primera.
# Fallback: si el DUNS no tiene match en df_proveedores (Nombre Proveedor = NaN),
# se usa el DUNS directamente como prefijo para garantizar que el campo nunca quede vacío.
_prefijo = df_detalle['Nombre Proveedor'].where(
    df_detalle['Nombre Proveedor'].notna(),
    df_detalle['DUNS Proveedor'].astype(str)  # fallback: usar DUNS si no hay nombre
).str.split().str[0]
df_detalle['ID Embalaje'] = _prefijo + '_' + df_detalle['Embalaje']

# ── Tabla 2: Resumen por Camión (derivado desde df_detalle) ─────────
resumen_rows = []
for camion in resultados:
    cam_id    = camion['id']
    df_cam    = df_detalle[df_detalle['Camión N°'] == cam_id]
    # Volumen calculado desde bloques físicos: por_largo × por_ancho × niveles × vol_unitario
    # Esto evita doble conteo cuando múltiples PNs comparten el mismo bloque (v34)
    vol_ocu   = round(sum(
        b['por_largo'] * b['por_ancho'] * b['niveles'] *
        b['largo_emb'] * b['ancho_emb'] * b['alto_emb'] / 1_000_000
        for b in camion['bloques']
        if not b.get('solapado', False)), 4)
    pct       = round(vol_ocu / (VOL_CAMION / 1_000_000) * 100, 1)
    largo_us  = camion['largo_usado']
    provs     = list(dict.fromkeys(b['duns'] for b in camion['bloques']))
    resumen_rows.append({
        'Camión N°':               cam_id,
        'Proveedores (DUNS)':      ', '.join(str(p) for p in provs),
        'Total Part Numbers':      int(df_cam['Part Number'].nunique()),
        'Total Embalajes':         int(df_cam['Cant. Embalajes'].sum()),
        'Superficie ocupada (m²)':     round(sum(
            b['largo_ocupado'] * b['por_ancho'] *
            (b['ancho_emb'] if b['orient_tipo'] == 'T' else b['largo_emb'])
            for b in camion['bloques']
            if not b.get('solapado', False)) / 10_000, 4),  # solapados no ocupan piso adicional
        'Sup. total camión (m²)':      round(CAMION_LARGO * CAMION_ANCHO / 10_000, 4),
        'Superficie ocupada %':        round(sum(
            b['largo_ocupado'] * b['por_ancho'] *
            (b['ancho_emb'] if b['orient_tipo'] == 'T' else b['largo_emb'])
            for b in camion['bloques']
            if not b.get('solapado', False)) / (CAMION_LARGO * CAMION_ANCHO) * 100, 1),  # solapados no ocupan piso adicional
        'Volumen ocupado (m³)':    vol_ocu,
        'Vol. camión (m³)':        round(VOL_CAMION / 1_000_000, 4),
        'Ocupación volumétrica %': pct
    })
df_resumen = pd.DataFrame(resumen_rows)


# ── Tabla 3: Resumen por Part Number ────────────────────────────────
# Usa df_pn (plan de carga a nivel PN) y parámetros del camión ya calculados
sup_camion_m2 = round(CAMION_LARGO * CAMION_ANCHO / 10_000, 4)
resumen_pn_rows = []
for _, r in df_pn.iterrows():
    alto_emb   = r['Alto']
    nivel_rest = r['NivelesApilamiento']
    niv_fis    = math.floor(CAMION_ALTO / alto_emb)
    niv_max    = min(niv_fis, int(nivel_rest)) if pd.notna(nivel_rest) else niv_fis
    cant_col   = math.ceil(r['CantidadEmbalajes'] / niv_max)
    sup_min    = round(cant_col * r['Largo'] * r['Ancho'] / 10_000, 4)
    cam_nec    = round(sup_min / sup_camion_m2, 1)
    resumen_pn_rows.append({
        'DUNS Proveedor':              r['DUNS'],
        'Part Number':                 r['PartNumber'],
        'Cantidad de Part Number':     r['CantidadPN'],
        'Embalaje':                    r['Embalaje'],
        'Cant. PNs por Embalaje':      r['PNsPorEmbalaje'],
        'Cant. Embalajes':             r['CantidadEmbalajes'],
        'Superficie Ocupada (m²)':     sup_min,
        'Camiones Necesarios':         cam_nec,
    })
df_resumen_pn = pd.DataFrame(resumen_pn_rows)

print('═' * 150)
print('  📋  RESUMEN POR PART NUMBER')
print('═' * 150)
display(df_resumen_pn)
# ── Mostrar tablas ───────────────────────────────────────────────────
for cam_id, grupo in df_detalle.groupby('Camión N°'):
    res = df_resumen[df_resumen['Camión N°'] == cam_id].iloc[0]
    print('═' * 150)
    print(
        f"  🚛  CAMIÓN N° {cam_id}  │  "
        f"Ocupación: {res['Ocupación volumétrica %']}%  │  "
        f"Vol. ocupado: {res['Volumen ocupado (m³)']} m³  │  "
        f"Proveedores DUNS: {res['Proveedores (DUNS)']}"
    )
    print('═' * 150)
    display(grupo.drop(columns='Camión N°').reset_index(drop=True))
    print()

print('═' * 150)
print('  📋  RESUMEN GENERAL POR CAMIÓN')
print('═' * 150)
display(df_resumen)



══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
  📋  RESUMEN POR PART NUMBER
══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════


,DUNS Proveedor,Part Number,Cantidad de Part Number,Embalaje,Cant. PNs por Embalaje,Cant. Embalajes,Superficie Ocupada (m²),Camiones Necesarios
0,111000001,PN-026,216,EMB-01,10,22,0.7200,0.0000
1,111000001,PN-006,78,EMB-01,10,8,0.2400,0.0000
2,111000001,PN-032,481,EMB-04,4,121,58.5600,1.4000
3,111000001,PN-044,443,EMB-11,8,56,4.9500,0.1000
4,111000001,PN-047,447,EMB-12,4,112,34.5800,0.9000
5,667000016,PN-087,3500,EMB-48,10,350,81.1800,2.0000
6,667000016,PN-088,5430,EMB-49,5,1086,353.9536,8.7000
7,999000009,PN-008,74,EMB-02,6,13,0.8000,0.0000
8,999000009,PN-067,53,EMB-39,5,11,5.4144,0.1000
9,999000009,PN-068,235,EMB-43,4,59,9.9684,0.2000


══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
  🚛  CAMIÓN N° 1  │  Ocupación: 60.9%  │  Vol. ocupado: 69.0424 m³  │  Proveedores DUNS: 667000016
══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════


,DUNS Proveedor,Part Number,Cant. Piezas (PN),Embalaje,Cant. Embalajes,Orientación L×A×H (cm),Orientación,Niveles (Alto),Unid. por Ancho,Unid. por Largo,Largo Ocupado (cm),Pos. Inicio (cm),Pos. Fin (cm),Volumen Ocupado (m³),Bloque Idx,Nombre Proveedor,ID Embalaje
0,667000016,PN-087,480,EMB-48,48,75 x 123 x 60,L,4,2,6,450,0,450,26.5680,0,Pi Usinagem de Precisão Ltda.,Pi_EMB-48
1,667000016,PN-088,240,EMB-49,48,143 x 91 x 68,T,4,2,6,858,450,1308,42.4744,1,Pi Usinagem de Precisão Ltda.,Pi_EMB-49



══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
  🚛  CAMIÓN N° 2  │  Ocupación: 60.9%  │  Vol. ocupado: 69.0424 m³  │  Proveedores DUNS: 667000016
══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════


...

,DUNS Proveedor,Part Number,Cant. Piezas (PN),Embalaje,Cant. Embalajes,Orientación L×A×H (cm),Orientación,Niveles (Alto),Unid. por Ancho,Unid. por Largo,Largo Ocupado (cm),Pos. Inicio (cm),Pos. Fin (cm),Volumen Ocupado (m³),Bloque Idx,Nombre Proveedor,ID Embalaje
0,111000001,PN-047,159,EMB-12,40,95 x 65 x 48,T,2,4,5,475,0,475,11.8560,0,Metalúrgica Alfa Ltda.,Metalúrgica_EMB-12



══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
  🚛  CAMIÓN N° 21  │  Ocupación: 36.5%  │  Vol. ocupado: 41.3458 m³  │  Proveedores DUNS: 999000009
══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════


,DUNS Proveedor,Part Number,Cant. Piezas (PN),Embalaje,Cant. Embalajes,Orientación L×A×H (cm),Orientación,Niveles (Alto),Unid. por Ancho,Unid. por Largo,Largo Ocupado (cm),Pos. Inicio (cm),Pos. Fin (cm),Volumen Ocupado (m³),Bloque Idx,Nombre Proveedor,ID Embalaje
0,999000009,PN-068,235,EMB-43,59,71 x 117 x 55,L,5,2,6,426,0,426,26.9562,0,Iota Sistemas de Freios S.A.,Iota_EMB-43
1,999000009,PN-067,53,EMB-39,11,144 x 94 x 71,T,3,2,2,288,426,714,10.5716,1,Iota Sistemas de Freios S.A.,Iota_EMB-39
2,999000009,PN-008,74,EMB-02,13,80 x 50 x 40,T,3,5,1,80,714,794,2.0800,2,Iota Sistemas de Freios S.A.,Iota_EMB-02



══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
  📋  RESUMEN GENERAL POR CAMIÓN
══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════


,Camión N°,Proveedores (DUNS),Total Part Numbers,Total Embalajes,Superficie ocupada (m²),Sup. total camión (m²),Superficie ocupada %,Volumen ocupado (m³),Vol. camión (m³),Ocupación volumétrica %
0,1,667000016,2,96,26.6856,40.5000,65.9000,69.0424,113.4000,60.9000
1,2,667000016,2,96,26.6856,40.5000,65.9000,69.0424,113.4000,60.9000
2,3,667000016,2,96,26.6856,40.5000,65.9000,69.0424,113.4000,60.9000
3,4,667000016,2,96,26.6856,40.5000,65.9000,69.0424,113.4000,60.9000
4,5,667000016,2,96,26.6856,40.5000,65.9000,69.0424,113.4000,60.9000
5,6,667000016,2,96,26.6856,40.5000,65.9000,69.0424,113.4000,60.9000
6,7,667000016,2,96,26.6856,40.5000,65.9000,69.0424,113.4000,60.9000
7,8,667000016,1,80,26.0260,40.5000,64.3000,70.7907,113.4000,62.4000
8,9,667000016,1,80,26.0260,40.5000,64.3000,70.7907,113.4000,62.4000
9,10,667000016,1,80,26.0260,40.5000,64.3000,70.7907,113.4000,62.4000


## 12. Validaciones de consistencia

Ejecuta cinco validaciones automáticas sobre los resultados del algoritmo para garantizar integridad de datos y cumplimiento de las reglas de negocio, e imprime un resumen de estadísticas de ocupación al final:

**Validación 1 — Total de piezas por PN:** verifica que la suma de piezas asignadas a cada Part Number en todos los camiones coincide exactamente con la demanda original.

**Validación 2 — Coherencia volumétrica:** verifica que el volumen del resumen por camión coincide con la suma de volúmenes del detalle por PN para ese mismo camión.

**Validación 3 — Coherencia de embalajes:** verifica que el total de embalajes del resumen coincide con la suma del detalle.

**Validación 4 — No mezcla de DUNS por tipo de embalaje:** verifica que en ningún camión existe intercalado longitudinal entre bloques de distintos proveedores. Un mismo tipo de embalaje puede aparecer en el mismo camión para distintos proveedores, siempre que sus zonas longitudinales no se intercalen.

**Validación 5 — Superposición vertical entre tipos distintos:** verifica que ningún par de bloques de distinto tipo comparte rango longitudinal, salvo la excepción permitida: mismo DUNS y mismas dimensiones de piso (L×A). Cualquier otra superposición entre tipos distintos se reporta como error.

**Estadísticas de ocupación:** al final imprime el promedio, máximo y mínimo de ocupación volumétrica sobre todos los camiones del corrido.

In [236]:
print('── Validación 1: total de piezas por PN vs demanda original ──')
pn_total = df_detalle.groupby(['DUNS Proveedor','Part Number'])['Cant. Piezas (PN)'].sum()
errores = 0
for _, r in df_pn.iterrows():
    try:
        total = pn_total.loc[(r['DUNS'], r['PartNumber'])]
    except:
        total = 0
    if total != r['CantidadPN']:
        print(f'  ❌ PN {r["PartNumber"]} DUNS {r["DUNS"]}: demanda={r["CantidadPN"]} asignado={total}')
        errores += 1
if errores == 0:
    print('  ✅ Todas las piezas de todos los PNs están correctamente asignadas')

print('\n── Validación 2: volumen del resumen == suma del detalle por camión ──')
errores = 0
for _, rr in df_resumen.iterrows():
    # Volumen recalculado desde bloques físicos (igual que en df_resumen) para que la comparación sea válida
    cam_bloques = next((c['bloques'] for c in resultados if c['id'] == rr['Camión N°']), [])
    vol_det = round(sum(
        b['por_largo'] * b['por_ancho'] * b['niveles'] *
        b['largo_emb'] * b['ancho_emb'] * b['alto_emb'] / 1_000_000
        for b in cam_bloques
        if not b.get('solapado', False)), 4)
    if abs(vol_det - rr['Volumen ocupado (m³)']) > 0.0001:
        print(f'  ❌ Camión {rr["Camión N°"]}: resumen={rr["Volumen ocupado (m³)"]} vs detalle={vol_det}')
        errores += 1
if errores == 0:
    print('  ✅ Volúmenes coherentes entre resumen y detalle')

print('\n── Validación 3: total embalajes (resumen == detalle) ──')
errores = 0
for _, rr in df_resumen.iterrows():
    emb_det = int(df_detalle[df_detalle['Camión N°']==rr['Camión N°']]['Cant. Embalajes'].sum())
    if emb_det != rr['Total Embalajes']:
        print(f'  ❌ Camión {rr["Camión N°"]}: resumen={rr["Total Embalajes"]} vs detalle={emb_det}')
        errores += 1
if errores == 0:
    print('  ✅ Totales de embalajes coherentes entre resumen y detalle')

print('\n── Validación 4: no intercalado longitudinal entre proveedores ──')
print('   Nota: un mismo tipo de embalaje puede coexistir en el mismo camión para distintos')
print('   proveedores (están etiquetados). Lo que se verifica es que sus zonas no se intercalen.')
errores = 0
for cam in resultados:
    provs_cam = set(b['duns'] for b in cam['bloques'])
    for duns in provs_cam:
        fin_duns  = max(b['pos_fin']    for b in cam['bloques'] if b['duns']==duns)
        ini_duns  = min(b['pos_inicio'] for b in cam['bloques'] if b['duns']==duns)
        for bo in cam['bloques']:
            if bo['duns'] != duns and ini_duns < bo['pos_inicio'] < fin_duns:
                print(f'  ❌ Camión {cam["id"]}: bloque de {bo["duns"]} intercalado en zona de {duns}')
                errores += 1
if errores == 0:
    print('  ✅ Sin intercalado — cada proveedor ocupa zona longitudinal continua')

print('\n── Validación 5: superposición vertical entre tipos de embalaje distintos ──')
print('   Regla general: dos tipos distintos no se apilan.')
print('   Excepción permitida: mismo DUNS y mismas dimensiones de piso (L×A).')
errores = 0
for cam in resultados:
    bloques = cam['bloques']
    for i in range(len(bloques)):
        for j in range(i + 1, len(bloques)):
            b1, b2 = bloques[i], bloques[j]
            solapa_long = b1['pos_inicio'] < b2['pos_fin'] and b2['pos_inicio'] < b1['pos_fin']
            if not solapa_long or b1['embalaje'] == b2['embalaje']:
                continue
            # Excepción: mismo DUNS y mismas dimensiones de piso → apilamiento mixto permitido
            mismo_duns = b1['duns'] == b2['duns']
            mismo_piso = {b1['largo_emb'], b1['ancho_emb']} == {b2['largo_emb'], b2['ancho_emb']}
            if mismo_duns and mismo_piso:
                continue  # permitido por la excepción de apilamiento mixto
            print(f'  ❌ Camión {cam["id"]}: {b1["embalaje"]} [{b1["pos_inicio"]}-{b1["pos_fin"]}]'
                  f' solapa con {b2["embalaje"]} [{b2["pos_inicio"]}-{b2["pos_fin"]}]'
                  f' (DUNS {b1["duns"]} vs {b2["duns"]})')
            errores += 1
if errores == 0:
    print('  ✅ Sin superposiciones no permitidas entre tipos de embalaje distintos.')
    print('     → Solo se solapan tipos del mismo DUNS con igual L×A (apilamiento mixto permitido).')

print(f'\n── Estadísticas de ocupación ──')
print(f'  Ocupación promedio : {df_resumen["Ocupación volumétrica %"].mean():.1f}%')
print(f'  Ocupación máxima   : {df_resumen["Ocupación volumétrica %"].max():.1f}%')
print(f'  Ocupación mínima   : {df_resumen["Ocupación volumétrica %"].min():.1f}%')

── Validación 1: total de piezas por PN vs demanda original ──
  ✅ Todas las piezas de todos los PNs están correctamente asignadas

── Validación 2: volumen del resumen == suma del detalle por camión ──
  ✅ Volúmenes coherentes entre resumen y detalle

── Validación 3: total embalajes (resumen == detalle) ──
  ✅ Totales de embalajes coherentes entre resumen y detalle

── Validación 4: no intercalado longitudinal entre proveedores ──
   Nota: un mismo tipo de embalaje puede coexistir en el mismo camión para distintos
   proveedores (están etiquetados). Lo que se verifica es que sus zonas no se intercalen.
  ✅ Sin intercalado — cada proveedor ocupa zona longitudinal continua

── Validación 5: superposición vertical entre tipos de embalaje distintos ──
   Regla general: dos tipos distintos no se apilan.
   Excepción permitida: mismo DUNS y mismas dimensiones de piso (L×A).
  ✅ Sin superposiciones no permitidas entre tipos de embalaje distintos.
     → Solo se solapan tipos del mismo DUNS 

## 13. Estadísticas de ocupación y análisis de causas

Esta celda realiza dos tareas:

**1 — Histogramas de ocupación:** genera dos gráficos de barras con líneas de referencia (capacidad máxima y promedio):
- **Histograma de volumen ocupado (m³):** distribución de los m³ cargados por camión
- **Histograma de superficie ocupada (m²):** distribución del área de piso ocupada por camión

Ambos histogramas se guardan en un buffer de imagen para embeberse en la **Hoja 6** del archivo de salida.

**2 — Análisis de causas del bottom 50%:** calcula el umbral del percentil 50 de ocupación volumétrica e identifica los camiones por debajo de ese umbral. Para cada uno ejecuta `_analizar_camion()`, que evalúa 7 criterios de diagnóstico y genera causas y recomendaciones específicas:
- **APILAMIENTO:** restricción de niveles por debajo del máximo físico posible
- **RESIDUO TRANSV.:** espacio transversal libre no aprovechable por dimensiones del embalaje
- **RESIDUO LONG.:** espacio longitudinal residual al final del camión sin carga disponible
- **LÍM. PROVEEDORES:** límite de 3 proveedores impidió incorporar más carga
- **ORIENT.:** orientación del embalaje subóptima respecto al largo real usado
- **EMBALAJE ÚNICO:** único tipo de embalaje del proveedor con baja cobertura de ancho y altura
- **DEMANDA:** demanda insuficiente del proveedor para llenar el camión

El resultado `_analisis_rows` se consume en la celda 14 (tabla en Hoja 6 del Excel) y en la celda 15 (panel Análisis Ocupación del dashboard HTML).

In [237]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import io

# Datos desde df_resumen ya calculado
volumenes   = df_resumen['Volumen ocupado (m³)'].astype(float).tolist()
superficies = df_resumen['Superficie ocupada (m²)'].astype(float).tolist()
vol_camion  = round(VOL_CAMION / 1_000_000, 4)
sup_camion  = round(CAMION_LARGO * CAMION_ANCHO / 10_000, 4)
n_camiones  = len(volumenes)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Estadísticas de Ocupación de Camiones', fontsize=14, fontweight='bold', y=1.01)

# ── Histograma 1: Volumen ocupado (m³) ──────────────────────────────
ax1 = axes[0]
ax1.bar(range(1, n_camiones + 1), volumenes, color='#2E75B6', edgecolor='white', linewidth=0.5)
ax1.axhline(vol_camion, color='#C00000', linestyle='--', linewidth=1.2,
            label=f'Cap. máx. camión: {vol_camion} m³')
ax1.axhline(sum(volumenes) / n_camiones, color='#ED7D31', linestyle=':', linewidth=1.2,
            label=f'Promedio: {sum(volumenes)/n_camiones:.2f} m³')
ax1.set_title('Volumen cargado por camión (m³)', fontsize=11, fontweight='bold')
ax1.set_xlabel('Camión N°', fontsize=10)
ax1.set_ylabel('m³', fontsize=10)
ax1.set_xticks(range(1, n_camiones + 1))
ax1.set_xticklabels([str(i) for i in range(1, n_camiones + 1)], fontsize=7)
ax1.set_ylim(0, vol_camion * 1.1)
ax1.legend(fontsize=8)
ax1.grid(axis='y', alpha=0.3)
ax1.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.1f'))

# ── Histograma 2: Superficie ocupada (m²) ───────────────────────────
ax2 = axes[1]
ax2.bar(range(1, n_camiones + 1), superficies, color='#548235', edgecolor='white', linewidth=0.5)
ax2.axhline(sup_camion, color='#C00000', linestyle='--', linewidth=1.2,
            label=f'Sup. total camión: {sup_camion} m²')
ax2.axhline(sum(superficies) / n_camiones, color='#ED7D31', linestyle=':', linewidth=1.2,
            label=f'Promedio: {sum(superficies)/n_camiones:.2f} m²')
ax2.set_title('Superficie ocupada por camión (m²)', fontsize=11, fontweight='bold')
ax2.set_xlabel('Camión N°', fontsize=10)
ax2.set_ylabel('m²', fontsize=10)
ax2.set_xticks(range(1, n_camiones + 1))
ax2.set_xticklabels([str(i) for i in range(1, n_camiones + 1)], fontsize=7)
ax2.set_ylim(0, sup_camion * 1.1)
ax2.legend(fontsize=8)
ax2.grid(axis='y', alpha=0.3)
ax2.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.1f'))

plt.tight_layout()

# Guardar imagen en buffer para embeber en Excel
_buf = io.BytesIO()
plt.savefig(_buf, format='png', dpi=150, bbox_inches='tight')
_buf.seek(0)
plt.show()
print('✅ Histogramas generados')

# ── Análisis del bottom 25%: causas y recomendaciones ───────────────
import math as _math
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

_umbral_pct = df_resumen['Ocupación volumétrica %'].quantile(0.50)

# Diccionario de causas y recomendaciones por camión (basado en análisis del algoritmo)
# Se calculan dinámicamente a partir de los datos reales de cada camión
def _analizar_camion(camion, df_apilamiento_orig, ids_prearmados=None):
    causas = []
    recomendaciones = []

    # ── Criterio 9: Camión prearmado con espacio no aprovechado ─────
    # Un camión prearmado se cierra exactamente con la carga de su plantilla.
    # Si tiene espacio libre, la causa es el prearmado mismo —no una ineficiencia
    # del algoritmo—, por lo que se reporta como categoría exclusiva y se omiten
    # los demás criterios para no generar análisis confusos sobre ese camión.
    if ids_prearmados and camion['id'] in ids_prearmados:
        residuo = CAMION_LARGO - camion['largo_usado']
        if residuo > 0:
            causas.append(
                f"Camión prearmado con espacio no aprovechado: el camión fue armado "
                f"según una plantilla de prearmado fija y no recibe carga adicional "
                f"aunque tenga {residuo} cm ({round(residuo/CAMION_LARGO*100)}%) "
                f"de longitud disponible. El espacio libre es una consecuencia "
                f"intencional de la restricción de prearmado."
            )
            recomendaciones.append(
                f"Evaluar si la plantilla de prearmado puede ajustarse para incluir "
                f"más embalajes del mismo proveedor y aprovechar los {residuo} cm "
                f"disponibles, o si la restricción de prearmado exacto puede "
                f"flexibilizarse para permitir carga complementaria en el espacio residual."
            )
            return causas, recomendaciones

    api = df_apilamiento_orig.copy()
    api.columns = ['DUNS','Embalaje','NivelesApilamiento']

    embalajes_vistos = set()
    for b in camion['bloques']:
        key = (b['duns'], b['embalaje'])
        if key in embalajes_vistos:
            continue
        embalajes_vistos.add(key)

        niv_fisico = _math.floor(CAMION_ALTO / b['alto_emb'])
        niv_rest_rows = api[(api['DUNS']==b['duns']) & (api['Embalaje']==b['embalaje'])]['NivelesApilamiento']
        tiene_rest = len(niv_rest_rows) > 0 and pd.notna(niv_rest_rows.values[0])

        if tiene_rest:
            niv_rest_val = int(niv_rest_rows.values[0])
            if niv_rest_val < niv_fisico:
                causas.append(
                    f"Restricción de apilamiento en {b['embalaje']} (DUNS {b['duns']}): "
                    f"se permiten solo {niv_rest_val} niveles de {niv_fisico} físicamente posibles "
                    f"({b['alto_emb']} cm alto × {niv_rest_val} = {b['alto_emb']*niv_rest_val} cm "
                    f"de {CAMION_ALTO} cm disponibles)."
                )
                recomendaciones.append(
                    f"Revisar con el proveedor DUNS {b['duns']} si la restricción de apilamiento "
                    f"de {b['embalaje']} puede incrementarse. Pasar de {niv_rest_val} a {niv_fisico} niveles "
                    f"aumentaría la ocupación volumétrica de este embalaje en un "
                    f"{round((niv_fisico/niv_rest_val - 1)*100)}%."
                )

        ao = b['ancho_emb'] if b['orient_tipo'] == 'T' else b['largo_emb']
        ancho_ocu = b['por_ancho'] * ao
        residuo_ancho = CAMION_ANCHO - ancho_ocu
        if residuo_ancho >= 30:
            causas.append(
                f"Residuo transversal en {b['embalaje']} (DUNS {b['duns']}): "
                f"el embalaje ocupa {ancho_ocu} cm de los {CAMION_ANCHO} cm de ancho disponibles "
                f"({b['por_ancho']} unidades × {ao} cm), dejando {residuo_ancho} cm libres "
                f"que no pueden ser aprovechados por no caber otra columna."
            )
            recomendaciones.append(
                f"Evaluar si existe otro embalaje del mismo proveedor con dimensiones ≤ {residuo_ancho} cm "
                f"de ancho que pueda colocarse en el espacio transversal libre de {b['embalaje']}. "
                f"De no existir, considerar negociar con el proveedor embalajes con ancho más próximo "
                f"a divisores exactos de {CAMION_ANCHO} cm (p.ej. {CAMION_ANCHO//3} cm o {CAMION_ANCHO//4} cm)."
            )

    residuo_largo = CAMION_LARGO - camion['largo_usado']
    if residuo_largo > 10:
        causas.append(
            f"Espacio longitudinal residual de {residuo_largo} cm al final del camión: "
            f"el algoritmo agotó los embalajes disponibles del proveedor y no encontró "
            f"embalajes de otros proveedores que cupiesen en ese espacio."
        )
        recomendaciones.append(
            f"Verificar si algún proveedor pendiente tiene embalajes con largo ≤ {residuo_largo} cm "
            f"que puedan incorporarse como carga complementaria. "
            f"Alternativamente, acumular demanda adicional del mismo proveedor antes de despachar el camión."
        )

    n_provs = len(set(b['duns'] for b in camion['bloques']))
    if n_provs == 3:
        causas.append(
            f"Camión con 3 proveedores distintos (límite máximo): la restricción de máximo "
            f"3 proveedores por camión impidió incorporar carga adicional de otros proveedores "
            f"que pudieran haber aprovechado el espacio residual."
        )
        recomendaciones.append(
            f"Evaluar si el límite de 3 proveedores por camión puede flexibilizarse en casos "
            f"donde el espacio residual supera el 30% del volumen total."
        )


    # ── Criterio 5: Orientación subóptima ──────────────────────────
    # Comprueba si la orientación contraria hubiera cargado más embalajes
    # dado el largo real que el camión terminó usando.
    largo_real = camion['largo_usado']
    lo_alt = b['ancho_emb'] if b['orient_tipo'] == 'T' else b['largo_emb']
    ao_alt = b['largo_emb'] if b['orient_tipo'] == 'T' else b['ancho_emb']
    niv_fisico_alt = _math.floor(CAMION_ALTO / b['alto_emb'])
    qa_alt = _math.floor(CAMION_ANCHO / ao_alt)
    ql_alt = _math.floor(largo_real / lo_alt) if lo_alt > 0 else 0
    qty_alt = niv_fisico_alt * qa_alt * ql_alt if qa_alt > 0 and ql_alt > 0 else 0
    lo_act = b['largo_emb'] if b['orient_tipo'] == 'L' else b['ancho_emb']
    ao_act = b['ancho_emb'] if b['orient_tipo'] == 'L' else b['largo_emb']
    ql_act = _math.floor(largo_real / lo_act) if lo_act > 0 else 0
    qa_act = _math.floor(CAMION_ANCHO / ao_act)
    qty_act = niv_fisico_alt * qa_act * ql_act if qa_act > 0 and ql_act > 0 else 0
    if qty_alt > qty_act and qty_act > 0:
        ganancia_pct = round((qty_alt / qty_act - 1) * 100)
        orient_actual  = f"{lo_act}×{ao_act} cm (largo×ancho)"
        orient_alterna = f"{lo_alt}×{ao_alt} cm (largo×ancho)"
        causas.append(
            f"Orientación subóptima de {b['embalaje']} (DUNS {b['duns']}): "
            f"la orientación usada ({orient_actual}) carga {qty_act} unidades dado el largo "
            f"real del camión ({largo_real} cm), mientras que la orientación contraria "
            f"({orient_alterna}) hubiera cargado {qty_alt} unidades, un {ganancia_pct}% más."
        )
        recomendaciones.append(
            f"Evaluar si el algoritmo puede considerar retroactivamente la orientación "
            f"{orient_alterna} para {b['embalaje']} del proveedor DUNS {b['duns']}. "
            f"Dado que la orientación se fija al colocar el primer bloque del embalaje, "
            f"una revisión post-optimización podría recuperar hasta {qty_alt - qty_act} "
            f"embalajes adicionales en este camión."
        )

    # ── Criterio 6: Embalaje con baja cobertura estructural ──────────
    # Un solo tipo de embalaje que no aprovecha bien ni el ancho ni la altura del camión.
    duns_presentes = list(set(b['duns'] for b in camion['bloques']))
    for duns_p in duns_presentes:
        bloques_prov = [b for b in camion['bloques'] if b['duns'] == duns_p]
        tipos_emb_prov = set(b['embalaje'] for b in bloques_prov)
        if len(tipos_emb_prov) == 1:
            b_rep = bloques_prov[0]
            ao_rep = b_rep['ancho_emb'] if b_rep['orient_tipo'] == 'T' else b_rep['largo_emb']
            cobertura_ancho = (b_rep['por_ancho'] * ao_rep) / CAMION_ANCHO
            niv_fisico_rep = _math.floor(CAMION_ALTO / b_rep['alto_emb'])
            cobertura_alto  = b_rep['niveles'] / niv_fisico_rep if niv_fisico_rep > 0 else 1
            if cobertura_ancho < 0.85 and cobertura_alto < 0.60:
                causas.append(
                    f"Baja cobertura estructural del embalaje {b_rep['embalaje']} "
                    f"(DUNS {duns_p}, único tipo de embalaje del proveedor en este camión): "
                    f"ocupa solo el {round(cobertura_ancho*100)}% del ancho "
                    f"({b_rep['por_ancho']} unidades × {ao_rep} cm de {CAMION_ANCHO} cm) "
                    f"y el {round(cobertura_alto*100)}% de la altura disponible "
                    f"({b_rep['niveles']} de {niv_fisico_rep} niveles físicos posibles). "
                    f"El embalaje no está dimensionado para aprovechar eficientemente el camión."
                )
                recomendaciones.append(
                    f"Analizar con el proveedor DUNS {duns_p} la posibilidad de redimensionar "
                    f"{b_rep['embalaje']} para que su ancho sea un divisor exacto o casi exacto "
                    f"de {CAMION_ANCHO} cm (por ejemplo {CAMION_ANCHO//2} cm o {CAMION_ANCHO//3} cm) "
                    f"y su alto permita más niveles sin restricción logística. "
                    f"Un embalaje mejor dimensionado podría aumentar la ocupación volumétrica "
                    f"en este camión en hasta un {round((1/cobertura_ancho * 1/cobertura_alto - 1)*100)}%."
                )

    # ── Criterio 7: Demanda insuficiente para llenar el camión ───────
    # El largo usado es menor al 60% del camión Y toda la demanda del proveedor
    # principal ya fue cargada (no es espacio vacío del algoritmo sino falta de carga).
    uso_largo_pct = camion['largo_usado'] / CAMION_LARGO
    if uso_largo_pct < 0.60:
        # Verificar que no hay pendiente del proveedor principal (ya se agotó toda la demanda)
        duns_principal = max(
            set(b['duns'] for b in camion['bloques']),
            key=lambda d: sum(b['largo_ocupado'] for b in camion['bloques'] if b['duns'] == d)
        )
        # Si el camión está poco lleno y no hay residuo longitudinal significativo
        # (el largo_usado ≈ largo real del último bloque), la causa es demanda insuficiente
        residuo_long = CAMION_LARGO - camion['largo_usado']
        if residuo_long <= 10:  # El camión terminó sin espacio vacío sustancial al final
            pass  # No aplica: el camión está lleno, el bajo volumen es por otra causa
        else:
            # Hay espacio vacío pero el largo usado es bajo: demanda ya agotada
            largo_libre_pct = round((1 - uso_largo_pct) * 100)
            causas.append(
                f"Demanda insuficiente del proveedor {duns_principal}: el camión solo usa "
                f"{round(uso_largo_pct*100)}% de la longitud disponible ({camion['largo_usado']} cm "
                f"de {CAMION_LARGO} cm) porque la demanda asignada a este envío fue menor a la "
                f"capacidad del camión, dejando {residuo_long} cm ({largo_libre_pct}%) sin carga."
            )
            recomendaciones.append(
                f"Evaluar si es posible acumular más demanda del proveedor {duns_principal} "
                f"antes de despachar el camión, o consolidar con otro proveedor geográficamente "
                f"cercano cuya carga pueda completar el {largo_libre_pct}% de longitud libre "
                f"({residuo_long} cm). Si la baja demanda es estructural, considerar negociar "
                f"una frecuencia de despacho menor con mayor volumen acumulado por envío."
            )

    # ── Criterio 8: Restricción de consolidado ─────────────────────
    # Si algún proveedor del camión tiene restricción 'Ninguno' o tiene
    # restricciones que limitan qué otros proveedores pueden acompañarlo,
    # eso puede explicar por qué el camión viaja con baja carga consolidada.
    duns_en_cam = list(set(b['duns'] for b in camion['bloques']))
    if len(duns_en_cam) == 1 and _restr:
        duns_solo = duns_en_cam[0]
        restr_val = _restr.get(duns_solo)
        if restr_val == 'Ninguno':
            causas.append(
                f"Restricción de consolidado del proveedor {duns_solo}: "
                f"el proveedor tiene restricción 'Ninguno', por lo que no puede "
                f"compartir camión con ningún otro proveedor. El espacio no utilizado "
                f"del camión no puede completarse con carga de otros proveedores."
            )
            recomendaciones.append(
                f"Revisar con el proveedor DUNS {duns_solo} si la restricción de "
                f"consolidado puede flexibilizarse para proveedores específicos, "
                f"al menos para aquellos de sectores no competitivos o con acuerdos "
                f"de confidencialidad. Alternativamente, acumular más demanda propia "
                f"antes de despachar para aprovechar mejor la capacidad del camión."
            )
        elif isinstance(restr_val, set) and restr_val:
            # Filtrar solo los DUNS excluidos que tienen demanda real en este envío.
            # Un DUNS sin demanda no podría haber sido carga complementaria de todas
            # formas, por lo que incluirlo en el comentario sería engañoso.
            duns_con_demanda = set(df_pn['DUNS'].unique()) - {duns_solo}
            restr_con_demanda = restr_val & duns_con_demanda
            if restr_con_demanda:
                causas.append(
                    f"Restricción de consolidado del proveedor {duns_solo}: "
                    f"el proveedor excluye consolidar con DUNS {', '.join(sorted(restr_con_demanda))}, "
                    f"lo que puede haber limitado las opciones de carga complementaria disponibles."
                )
                recomendaciones.append(
                    f"Verificar si la restricción de consolidado del proveedor {duns_solo} "
                    f"con los proveedores {', '.join(sorted(restr_con_demanda))} es estrictamente "
                    f"necesaria o puede revisarse para aumentar las opciones de consolidado "
                    f"y mejorar la ocupación del camión."
                )

    return causas, recomendaciones


# Generar tabla de análisis para el Excel
_bottom_rows = df_resumen[df_resumen['Ocupación volumétrica %'] <= _umbral_pct].sort_values('Ocupación volumétrica %')

_analisis_rows = []
for _, rr in _bottom_rows.iterrows():
    cam = next(c for c in resultados if c['id'] == rr['Camión N°'])
    causas, recomendaciones = _analizar_camion(cam, df_apilamiento, _ids_prearmados_global if '_ids_prearmados_global' in dir() else [])
    _analisis_rows.append({
        'cam_id':         rr['Camión N°'],
        'pct_vol':        rr['Ocupación volumétrica %'],
        'pct_sup':        rr['Superficie ocupada %'],
        'proveedores':    rr['Proveedores (DUNS)'],
        'causas':         causas,
        'recomendaciones': recomendaciones,
    })

print(f'✅ Análisis bottom 50% completado — umbral: {_umbral_pct:.1f}%')
print(f'   Camiones analizados: {len(_analisis_rows)}')
for r in _analisis_rows:
    print(f"\n   Camión {r['cam_id']} ({r['pct_vol']}% vol):")
    for c in r['causas']:
        print(f"     CAUSA: {c[:80]}...")


✅ Histogramas generados
✅ Análisis bottom 50% completado — umbral: 60.9%
   Camiones analizados: 12

   Camión 20 (10.5% vol):
     CAUSA: Restricción de apilamiento en EMB-12 (DUNS 111000001): se permiten solo 2 nivele...
     CAUSA: Espacio longitudinal residual de 1025 cm al final del camión: el algoritmo agotó...
     CAUSA: Demanda insuficiente del proveedor 111000001: el camión solo usa 32% de la longi...

   Camión 19 (34.1% vol):
     CAUSA: Restricción de apilamiento en EMB-04 (DUNS 111000001): se permiten solo 2 nivele...
     CAUSA: Residuo transversal en EMB-04 (DUNS 111000001): el embalaje ocupa 240 cm de los ...
     CAUSA: Restricción de apilamiento en EMB-12 (DUNS 111000001): se permiten solo 2 nivele...
     CAUSA: Espacio longitudinal residual de 45 cm al final del camión: el algoritmo agotó l...

   Camión 21 (36.5% vol):
     CAUSA: Residuo transversal en EMB-43 (DUNS 999000009): el embalaje ocupa 234 cm de los ...
     CAUSA: Residuo transversal en EMB-39 (DUNS 999

C:\Users\PC\AppData\Local\Temp\ipykernel_21064\3524674108.py:57: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 14. Exportar resultados a Excel

Genera el archivo `resultado_optimizacion.xlsx` con 11 hojas:

**Hojas de output** (color verde):
- Hoja 1 — Detalle por PN y Camion
- Hoja 2 — Resumen por Camion
- Hoja 3 — Resumen por Part Number
- Hoja 4 — Superficie minima por PN
- Hoja 5 — Matriz Distancias Proveedores
- Hoja 6 — Estadísticas de Ocupación (histogramas de volumen y superficie + tabla de análisis de causas del bottom 50%)

**Hojas de input** (color gris):
- Hoja 7 — tabla_embalajes_input
- Hoja 8 — tabla_demanda_input
- Hoja 9 — tabla_apilamiento_input
- Hoja 10 — tabla_camion_input
- Hoja 11 — tabla_proveedores_input


## 15. Cronograma de colectas

Genera el cronograma semanal de colectas a partir de dos inputs:
- `tabla_ventanas_colecta.xlsx`: ventanas horarias disponibles por proveedor y día
- `df_resumen`: resultado del algoritmo de optimización (proveedores por camión)

**Lógica de asignación:**
Para cada camión se identifican los proveedores que debe recolectar. Se busca el primer día
de la semana (Lunes a Domingo) en que **todos** los proveedores del camión tengan ventana
disponible y puedan atenderse en secuencia, considerando:
- **Duración de carga:** 1 hora fija por proveedor
- **Velocidad de traslado:** 35 km/h entre proveedores consecutivos
- **Orden de visita:** por ventana de inicio más temprana ese día

Si no existe un día con ventanas coincidentes para todos, los proveedores se asignan a días
distintos (un día por proveedor, eligiendo el primero disponible para cada uno).

Si un proveedor no tiene ninguna ventana disponible que permita la colecta, se propone una
**ventana sugerida** (marcada con ⚠️) calculada como la hora mínima de llegada posible,
independientemente de las ventanas de la tabla.

**Output:** `df_cronograma` con una fila por colecta, exportado como hoja adicional en el Excel.

In [238]:
import math as _math
from collections import defaultdict as _defaultdict

# ── Cargar tabla de ventanas de colecta ─────────────────────────────
df_ventanas = pd.read_excel('tabla_ventanas_colecta.xlsx',
                             engine='openpyxl', header=0)
_df_ventanas_raw = df_ventanas.copy()  # snapshot recién leído del Excel
df_ventanas.columns = ['DUNS','Nombre','Dia','H_Ini','H_Fin','N_Ventana','Obs']
df_ventanas = df_ventanas.dropna(subset=['DUNS','Dia'])
df_ventanas['DUNS'] = df_ventanas['DUNS'].astype(str)

def _to_min(t):
    """Convierte HH:MM (str o time) a minutos desde medianoche."""
    if isinstance(t, str):
        h, m = map(int, t.split(':'))
    else:
        h, m = t.hour, t.minute
    return h * 60 + m

def _to_hhmm(m):
    """Convierte minutos desde medianoche a string HH:MM."""
    return f'{int(m)//60:02d}:{int(m)%60:02d}'

# Construir diccionario: ventanas[duns][dia] = [(ini_min, fin_min, obs)]
_ventanas = _defaultdict(lambda: _defaultdict(list))
for _, row in df_ventanas.iterrows():
    _ventanas[str(row['DUNS'])][row['Dia']].append((
        _to_min(row['H_Ini']),
        _to_min(row['H_Fin']),
        str(row['Obs']) if pd.notna(row['Obs']) else ''
    ))

# ── Parámetros ───────────────────────────────────────────────────────
_CARGA_MIN  = 60    # duración de carga por proveedor (minutos)
_VELOCIDAD  = 35    # km/h para traslado entre proveedores
_DIAS_SEM   = ['Lunes','Martes','Miércoles','Jueves','Viernes']

def _transit_min(duns_a, duns_b):
    """
    Calcula el tiempo de traslado en minutos entre dos proveedores.

    Usa la matriz de distancias Haversine (df_matriz) y la velocidad
    de referencia de 35 km/h. Resultado redondeado hacia arriba.

    Args:
        duns_a: DUNS del proveedor de origen.
        duns_b: DUNS del proveedor de destino.

    Returns:
        int: Minutos de traslado redondeados al alza.
    """
    try:
        dist = float(df_matriz.loc[int(duns_a), int(duns_b)])
    except Exception:
        try:
            dist = float(df_matriz.loc[duns_a, duns_b])
        except Exception:
            dist = 0.0
    return _math.ceil(dist / _VELOCIDAD * 60)

# ── Registro de slots de ventana ya ocupados ─────────────────────────
# Clave: (duns, dia, ini_min) → cantidad de camiones ya asignados
# Cada ventana solo puede recibir un camión.
_slots_usados = _defaultdict(int)

def _slot_disponible(duns, dia, ini_min):
    """True si la ventana (duns, dia, ini_min) aún no tiene camión asignado."""
    return _slots_usados[(duns, dia, ini_min)] == 0

def _ocupar_slot(duns, dia, ini_min):
    """Marca la ventana como ocupada."""
    _slots_usados[(duns, dia, ini_min)] += 1

def _nombre_corto(duns):
    """Devuelve la primera palabra del nombre del proveedor."""
    vals = df_proveedores[df_proveedores['DUNS'].astype(str) == duns]['Nombre'].values
    if not len(vals):
        return duns
    return str(vals[0]).split()[0]

def _intentar_dia(duns_list, dia):
    """
    Intenta asignar todos los proveedores del camión a un mismo día,
    respetando que cada slot de ventana solo puede recibir un camión.

    Args:
        duns_list (list[str]): Lista de DUNS del camión.
        dia (str): Día de la semana a evaluar.

    Returns:
        list[dict] | None: Secuencia de colectas o None si no es factible.
    """
    avail = []
    for duns in duns_list:
        wins = [w for w in _ventanas[duns].get(dia, [])
                if _slot_disponible(duns, dia, w[0])]
        if not wins:
            return None
        avail.append((duns, sorted(wins, key=lambda w: w[0])))

    avail.sort(key=lambda x: x[1][0][0])

    cursor = avail[0][1][0][0]
    sequence = []

    for idx, (duns, wins) in enumerate(avail):
        placed = False
        for win in wins:
            win_ini, win_fin, obs = win
            if not _slot_disponible(duns, dia, win_ini):
                continue
            start = max(cursor, win_ini)
            end   = start + _CARGA_MIN
            if end <= win_fin:
                transit = 0
                if idx < len(avail) - 1:
                    transit = _transit_min(duns, avail[idx + 1][0])
                sequence.append({
                    'duns': duns, 'ini': start, 'fin': end,
                    'transit': transit, 'obs': obs, 'tipo': '✅',
                    'win_ini': win_ini
                })
                cursor = end + transit
                placed = True
                break
        if not placed:
            return None

    return sequence

# ── Algoritmo de programación con distribución equilibrada Lun–Vie ───
# Distribuye los camiones lo más uniformemente posible entre los 5 días.
# Cada slot de ventana (proveedor + día + hora) acepta solo un camión.
# Los camiones sin ventana disponible reciben un horario sugerido.

_schedule_rows = []
_colectas_por_dia = {d: 0 for d in _DIAS_SEM}

def _mejor_dia(duns_list):
    """
    Elige el día menos cargado donde todos los proveedores tienen
    slot disponible y la secuencia de colectas es factible.

    Returns:
        (dia, seq, tipo): tipo '✅' si hay ventana real, '⚠️' si sugerida.
    """
    dias_ord = sorted(_DIAS_SEM, key=lambda d: _colectas_por_dia[d])
    for dia in dias_ord:
        seq = _intentar_dia(duns_list, dia)
        if seq is not None:
            return dia, seq, '✅'
    return None, None, '⚠️'

def _dia_para_duns(duns):
    """
    Día menos cargado con slot disponible para este DUNS.
    Si no hay ninguno, devuelve el día menos cargado como sugerido.

    Returns:
        (dia, tiene_ventana, win_ini_min)
    """
    dias_ord = sorted(_DIAS_SEM, key=lambda d: _colectas_por_dia[d])
    for dia in dias_ord:
        for win in sorted(_ventanas[duns].get(dia, []), key=lambda w: w[0]):
            if _slot_disponible(duns, dia, win[0]):
                return dia, True, win
    # Sin slot disponible en ningún día → horario sugerido
    # Estimar el mejor horario: día menos cargado, hora de inicio de jornada
    return dias_ord[0], False, (8 * 60, 9 * 60, '')

def _nota_consolidado(duns_propio, otros_duns, dia, hora_ini):
    """
    Genera la nota de camión consolidado indicando los otros proveedores,
    el día y la hora de colecta de cada uno.
    """
    partes = []
    for d in otros_duns:
        if d != duns_propio:
            partes.append(f"{_nombre_corto(d)} ({dia} {hora_ini})")
    if not partes:
        return ''
    return 'Consolidado con: ' + ', '.join(partes)

for _, res_row in df_resumen.iterrows():
    cam_id    = res_row['Camión N°']
    provs_str = res_row['Proveedores (DUNS)']
    duns_list = [d.strip() for d in str(provs_str).split(',')]

    if len(duns_list) == 1:
        # ── Camión con un solo proveedor ────────────────────────────
        duns = duns_list[0]
        nombre = df_proveedores[df_proveedores['DUNS'].astype(str)==duns]['Nombre'].values
        nombre = str(nombre[0]) if len(nombre) else duns

        dia, tiene_ventana, win = _dia_para_duns(duns)
        win_ini, win_fin, obs = win
        if tiene_ventana:
            _ocupar_slot(duns, dia, win_ini)
            _schedule_rows.append({
                'Camión N°': cam_id, 'Día': dia, 'Orden': 1,
                'DUNS': duns, 'Proveedor': nombre,
                'Hora Inicio': _to_hhmm(win_ini),
                'Hora Fin':    _to_hhmm(win_ini + _CARGA_MIN),
                'Traslado al siguiente (min)': 0,
                'Tipo': '✅', 'Nota': obs
            })
        else:
            _schedule_rows.append({
                'Camión N°': cam_id, 'Día': dia, 'Orden': 1,
                'DUNS': duns, 'Proveedor': nombre,
                'Hora Inicio': _to_hhmm(win_ini),
                'Hora Fin':    _to_hhmm(win_ini + _CARGA_MIN),
                'Traslado al siguiente (min)': 0,
                'Tipo': '⚠️', 'Nota': ''
            })
        _colectas_por_dia[dia] += 1

    else:
        # ── Camión con múltiples proveedores ────────────────────────
        dia_comun, seq, tipo = _mejor_dia(duns_list)

        if tipo == '✅':
            # Marcar todos los slots como ocupados
            for s in seq:
                _ocupar_slot(s['duns'], dia_comun, s['win_ini'])
            # Generar notas de consolidado para cada colecta
            for order, s in enumerate(seq, 1):
                duns = s['duns']
                nombre = df_proveedores[df_proveedores['DUNS'].astype(str)==duns]['Nombre'].values
                nombre = str(nombre[0]) if len(nombre) else duns
                otros = [x for x in seq if x['duns'] != duns]
                nota_cons = _nota_consolidado(
                    duns, [x['duns'] for x in otros],
                    dia_comun, _to_hhmm(otros[0]['ini']) if otros else ''
                ) if otros else s['obs']
                _schedule_rows.append({
                    'Camión N°': cam_id, 'Día': dia_comun, 'Orden': order,
                    'DUNS': duns, 'Proveedor': nombre,
                    'Hora Inicio': _to_hhmm(s['ini']),
                    'Hora Fin':    _to_hhmm(s['fin']),
                    'Traslado al siguiente (min)': 0,
                    'Tipo': '✅', 'Nota': nota_cons
                })
            _colectas_por_dia[dia_comun] += len(seq)
        else:
            # Sin día común → cada proveedor al día/slot disponible
            for order, duns in enumerate(duns_list, 1):
                nombre = df_proveedores[df_proveedores['DUNS'].astype(str)==duns]['Nombre'].values
                nombre = str(nombre[0]) if len(nombre) else duns
                dia, tiene_ventana, win = _dia_para_duns(duns)
                win_ini, win_fin, obs = win
                if tiene_ventana:
                    _ocupar_slot(duns, dia, win_ini)
                    _schedule_rows.append({
                        'Camión N°': cam_id, 'Día': dia, 'Orden': order,
                        'DUNS': duns, 'Proveedor': nombre,
                        'Hora Inicio': _to_hhmm(win_ini),
                        'Hora Fin':    _to_hhmm(win_ini + _CARGA_MIN),
                        'Traslado al siguiente (min)': 0,
                        'Tipo': '⚠️', 'Nota': ''
                    })
                else:
                    _schedule_rows.append({
                        'Camión N°': cam_id, 'Día': dia, 'Orden': order,
                        'DUNS': duns, 'Proveedor': nombre,
                        'Hora Inicio': _to_hhmm(win_ini),
                        'Hora Fin':    _to_hhmm(win_ini + _CARGA_MIN),
                        'Traslado al siguiente (min)': 0,
                        'Tipo': '⚠️', 'Nota': ''
                    })
                _colectas_por_dia[dia] += 1

df_cronograma = pd.DataFrame(_schedule_rows, columns=[
    'Camión N°','Día','Orden','DUNS','Proveedor',
    'Hora Inicio','Hora Fin','Traslado al siguiente (min)','Tipo','Nota'
])
_dia_orden = {d: i for i, d in enumerate(_DIAS_SEM)}
df_cronograma['_dia_ord'] = df_cronograma['Día'].map(_dia_orden)
df_cronograma = df_cronograma.sort_values(
    ['_dia_ord', 'Camión N°', 'Hora Inicio']
).reset_index(drop=True)
df_cronograma['Orden'] = (
    df_cronograma.groupby(['Camión N°', 'Día']).cumcount() + 1
)
df_cronograma = df_cronograma.drop(columns='_dia_ord').reset_index(drop=True)

n_conflict = (df_cronograma['Tipo'] == '⚠️').sum()
n_ok       = (df_cronograma['Tipo'] == '✅').sum()
print(f'✅ Cronograma de colectas generado — {len(df_cronograma)} colectas programadas')
print(f'   ✅ Confirmadas: {n_ok}  |  ⚠️ Ventanas sugeridas: {n_conflict}')
display(df_cronograma)


✅ Cronograma de colectas generado — 22 colectas programadas
   ✅ Confirmadas: 15  |  ⚠️ Ventanas sugeridas: 7


,Camión N°,Día,Orden,DUNS,Proveedor,Hora Inicio,Hora Fin,Traslado al siguiente (min),Tipo,Nota
0,1,Lunes,1,667000016,Pi Usinagem de Precisão Ltda.,07:00,08:00,0,✅,
1,4,Lunes,1,667000016,Pi Usinagem de Precisão Ltda.,08:00,09:00,0,✅,
2,7,Lunes,1,667000016,Pi Usinagem de Precisão Ltda.,09:00,10:00,0,✅,
3,10,Lunes,1,667000016,Pi Usinagem de Precisão Ltda.,10:00,11:00,0,✅,
4,17,Lunes,1,111000001,Metalúrgica Alfa Ltda.,08:00,09:00,0,⚠️,
5,18,Lunes,1,111000001,Metalúrgica Alfa Ltda.,09:00,10:00,0,✅,
6,19,Lunes,1,111000001,Metalúrgica Alfa Ltda.,10:00,11:00,0,✅,
7,20,Lunes,1,111000001,Metalúrgica Alfa Ltda.,11:00,12:00,0,✅,
8,12,Martes,1,667000016,Pi Usinagem de Precisão Ltda.,08:00,09:00,0,⚠️,
9,14,Martes,1,667000016,Pi Usinagem de Precisão Ltda.,08:00,09:00,0,⚠️,


## 16. Generar Dashboard HTML

Genera el archivo `dashboard_ocupacion_camiones.html` — un dashboard interactivo autocontenido con todos los datos del análisis, que puede abrirse en cualquier navegador sin conexión a internet.

El dashboard incluye 10 paneles con navegación por tabs:
- **Resumen:** gráficos de superficie %, volumen % y brecha por camión + tabla resumen
- **Superficie / Volumen:** análisis detallado de ocupación con gráficos de dispersión
- **Restricciones:** altura utilizada por embalaje con restricción de apilamiento
- **Proveedores:** métricas y comparativas por proveedor
- **Distancias:** distribución de distancias, pares más cercanos y mapa de calor
- **Detalle Carga:** tabla filtrable con todos los registros de carga por PN y camión
- **Sup. Mínima PN / Resumen PN:** análisis a nivel de Part Number
- **Análisis Ocupación:** camiones del bottom 50% con sus causas expandibles al hacer clic y filtro por tipo de causa (APILAMIENTO / RESIDUO TRANSV. / RESIDUO LONG. / LÍM. PROVEEDORES / ORIENT. / EMBALAJE ÚNICO / DEMANDA)
- **Inputs:** tablas de datos de entrada (embalajes, demanda, apilamiento, camión, proveedores con dirección y coordenadas)

In [239]:
import json as _json
import math as _math

# ── Preparar datos para el dashboard ────────────────────────────────
_D = {}

# KPIs
_D['n_cam']      = len(df_resumen)
_D['sup_avg']    = round(df_resumen['Superficie ocupada %'].mean(), 1)
_D['sup_min']    = round(df_resumen['Superficie ocupada %'].min(), 1)
_D['sup_max']    = round(df_resumen['Superficie ocupada %'].max(), 1)
_D['vol_avg']    = round(df_resumen['Ocupación volumétrica %'].mean(), 1)
_D['vol_min']    = round(df_resumen['Ocupación volumétrica %'].min(), 1)
_D['vol_max']    = round(df_resumen['Ocupación volumétrica %'].max(), 1)
_D['gap_avg']    = round((df_resumen['Superficie ocupada %'] - df_resumen['Ocupación volumétrica %']).mean(), 1)
_D['cam_low']    = int((df_resumen['Superficie ocupada %'] < 70).sum())
_D['sup_tot_cam']= round(CAMION_LARGO * CAMION_ANCHO / 10_000, 1)
_D['camion']     = {'largo': CAMION_LARGO, 'ancho': CAMION_ANCHO, 'alto': CAMION_ALTO}

# Series por camión
_D['labels']       = [str(i) for i in df_resumen['Camión N°'].tolist()]
_D['sup']          = df_resumen['Superficie ocupada %'].round(1).tolist()
_D['vol']          = df_resumen['Ocupación volumétrica %'].round(1).tolist()
_D['gap']          = (df_resumen['Superficie ocupada %'] - df_resumen['Ocupación volumétrica %']).round(1).tolist()
_D['sup_m2']       = df_resumen['Superficie ocupada (m²)'].round(2).tolist()
_D['vol_m3']       = df_resumen['Volumen ocupado (m³)'].round(2).tolist()
_D['provs_by_cam'] = df_resumen['Proveedores (DUNS)'].tolist()

# Restricciones de apilamiento
_api = df_apilamiento.copy()
_emb_dims = df_embalajes[['DUNS','Embalaje','Alto']].drop_duplicates(subset=['DUNS','Embalaje'])
_api_m = _api.merge(_emb_dims, on=['DUNS','Embalaje'], how='left').dropna(subset=['Alto'])
_api_m['_alt_usa'] = _api_m['Alto'] * _api_m['NivelesApilamiento']
_api_m['_alt_pct'] = (_api_m['_alt_usa'] / CAMION_ALTO * 100).round(1)
_api_m['_label']   = _api_m['Embalaje'] + ' (' + _api_m['DUNS'].astype(str).str[-5:] + ')'
_api_m = _api_m.sort_values('_alt_usa')
_D['alt_labels'] = _api_m['_label'].tolist()
_D['alt_usa']    = _api_m['_alt_usa'].tolist()
_D['alt_pct']    = _api_m['_alt_pct'].tolist()

# Proveedores
_prov_stats = []
for _, _pr in df_proveedores.iterrows():
    _duns     = str(_pr['DUNS'])  # normalizar a str: df_detalle hereda str de la coerción de sección 3
    _cam_p = df_resumen[df_resumen['Proveedores (DUNS)'].astype(str).str.contains(_duns)]
    _det_p = df_detalle[df_detalle['DUNS Proveedor'] == _duns]
    _prov_stats.append({
        'name':   str(_pr['Nombre']).replace(' Ltda.','').replace(' S.A.',''),
        'dir':    str(_pr['Direccion']),
        'lat':    round(float(_pr['Latitud']),  6),
        'lon':    round(float(_pr['Longitud']), 6),
        'emb':    int(_det_p['Cant. Embalajes'].sum()),
        'cam':    int(_cam_p['Camión N°'].nunique()),
        'vol':    round(_cam_p['Ocupación volumétrica %'].mean(), 1) if len(_cam_p) > 0 else 0,
        'sup':    round(_cam_p['Superficie ocupada %'].mean(), 1) if len(_cam_p) > 0 else 0,
        'pzas':   int(_det_p['Cant. Piezas (PN)'].sum()),
        'vol_m3': round(_det_p['Volumen Ocupado (m³)'].sum(), 2),
        'n_pns':  int(_det_p['Part Number'].nunique()),
    })
_D['prov_stats'] = _prov_stats
_D['prov_names'] = [p['name'] for p in _prov_stats]
_D['duns_order'] = [str(d) for d in df_proveedores['DUNS'].tolist()]

# Matriz de distancias
_D['dist_matrix'] = df_matriz.values.tolist()

# Detalle de carga
_D['det_rows'] = []
for _, _r in df_detalle.iterrows():
    _D['det_rows'].append({
        'cam': int(_r['Camión N°']), 'duns': str(_r['DUNS Proveedor']),
        'nombre': str(_r['Nombre Proveedor']) if pd.notna(_r['Nombre Proveedor']) else '',
        'id_emb': str(_r['ID Embalaje']) if pd.notna(_r['ID Embalaje']) else '',
        'pn': _r['Part Number'], 'pzas': int(_r['Cant. Piezas (PN)']),
        'emb': _r['Embalaje'], 'cant_emb': int(_r['Cant. Embalajes']),
        'orient': _r['Orientación L×A×H (cm)'], 'orient_t': _r['Orientación'],
        'niv': int(_r['Niveles (Alto)']), 'por_ancho': int(_r['Unid. por Ancho']),
        'por_largo': int(_r['Unid. por Largo']), 'largo_ocu': int(_r['Largo Ocupado (cm)']),
        'pos_ini': int(_r['Pos. Inicio (cm)']), 'pos_fin': int(_r['Pos. Fin (cm)']),
        'vol': round(float(_r['Volumen Ocupado (m³)']), 4),
        'bloque_idx': int(_r['Bloque Idx']),
        'solapado': bool(_r['Solapado']),
    })

# Superficie mínima por PN
_D['sup_rows'] = []
for _, _r in df_sup_minima.iterrows():
    _D['sup_rows'].append({
        'duns': str(_r['DUNS Proveedor']), 'pn': _r['Part Number'],
        'dem': int(_r['Cantidad de Demanda (PN)']), 'emb': _r['Embalaje'],
        'cant_emb': int(_r['Cant. Unidades de Embalaje']),
        'largo': int(_r['Largo Embalaje (cm)']), 'ancho': int(_r['Ancho Embalaje (cm)']),
        'alto': int(_r['Alto Embalaje (cm)']), 'cols': int(_r['Cant. Columnas de Embalaje']),
        'sup_col': round(float(_r['Sup. por Columna (m²)']), 4),
        'sup_min': round(float(_r['Sup. Mínima Necesaria (m²)']), 4),
        'long_req': int(_r['Longitud Requerida (cm)']),
        'cam_nec': round(float(_r['Camiones Necesarios']), 1),
    })

# Resumen por PN
_D['rpn_rows'] = []
for _, _r in df_resumen_pn.iterrows():
    _D['rpn_rows'].append({
        'duns': str(_r['DUNS Proveedor']), 'pn': _r['Part Number'],
        'dem': int(_r['Cantidad de Part Number']), 'emb': _r['Embalaje'],
        'pns_por_emb': int(_r['Cant. PNs por Embalaje']),
        'cant_emb': int(_r['Cant. Embalajes']),
        'sup': round(float(_r['Superficie Ocupada (m²)']), 4),
        'cam_nec': round(float(_r['Camiones Necesarios']), 1),
    })

# Análisis bottom 50% para el dashboard
_D['analisis'] = []
for _ar in _analisis_rows:
    _D['analisis'].append({
        'cam':            _ar['cam_id'],
        'vol':            _ar['pct_vol'],
        'sup':            _ar['pct_sup'],
        'provs':          _ar['proveedores'],
        'causas':         _ar['causas'],
        'recomendaciones': _ar['recomendaciones'],
    })

# Cronograma de colectas para el dashboard
_DIAS_ORD = ['Lunes','Martes','Miércoles','Jueves','Viernes','Sábado','Domingo']
_D['cronograma'] = []
for _, _cr in df_cronograma.iterrows():
    _D['cronograma'].append({
        'cam':     int(_cr['Camión N°']),
        'dia':     _cr['Día'],
        'dia_ord': _DIAS_ORD.index(_cr['Día']) if _cr['Día'] in _DIAS_ORD else 7,
        'orden':   int(_cr['Orden']),
        'duns':    str(_cr['DUNS']),
        'prov':    _cr['Proveedor'],
        'ini':     _cr['Hora Inicio'],
        'fin':     _cr['Hora Fin'],
        'transit': int(_cr['Traslado al siguiente (min)']),
        'tipo':    _cr['Tipo'],
        'nota':    str(_cr['Nota']) if pd.notna(_cr['Nota']) else ''
    })

# ── Tablas de input originales (sin procesar) ──────────────────
def _df_to_records(df):
    """Convierte un DataFrame a lista de dicts con valores serializables."""
    import datetime as _dt
    def _ser(v):
        if isinstance(v, float) and _math.isnan(v): return None
        if isinstance(v, (int, float)) and not isinstance(v, bool): return str(v)
        if isinstance(v, (_dt.time, _dt.datetime)): return v.strftime('%H:%M')
        if isinstance(v, _dt.date): return v.isoformat()
        return v
    return [{str(k): _ser(v) for k, v in row.items()}
            for row in df.to_dict('records')]

_D['inp_embalajes']   = _df_to_records(_df_embalajes_raw)
_D['inp_demanda']     = _df_to_records(_df_demanda_raw)
_D['inp_apilamiento'] = _df_to_records(_df_apilamiento_raw)
_D['inp_camion']      = _df_to_records(_df_camion_raw)
_D['inp_proveedores'] = _df_to_records(_df_proveedores_raw)
_D['inp_paletizados']    = _df_to_records(_df_paletizados_raw)
_D['inp_ventanas']       = _df_to_records(_df_ventanas_raw)
_D['inp_restricciones']  = _df_to_records(_df_restricciones_raw)
# IDs de camiones que fueron generados por prearmado
# _ids_prearmados_global contiene los IDs reales de camiones generados por prearmado
_D['cams_prearmados'] = sorted(_ids_prearmados_global) if '_ids_prearmados_global' in dir() else []

# ── Leer template HTML e inyectar datos ────────────────────────────
_DATA_JS = _json.dumps(_D, ensure_ascii=False)

_html_tmpl = open('dashboard_template.html', 'r', encoding='utf-8').read()
_html_out  = _html_tmpl.replace('/*INJECT_DATA*/', 'const D = ' + _DATA_JS + ';')
# Reemplazar fuente IBM Plex por Overpass embebida en base64 (sin dependencia de internet)
import base64 as _b64
_overpass_light  = _b64.b64encode(open('overpass-light.woff2',  'rb').read()).decode()
_overpass_semi   = _b64.b64encode(open('overpass-semibold.woff2','rb').read()).decode()
_overpass_bold   = _b64.b64encode(open('overpass-bold.woff2',   'rb').read()).decode()
_font_faces = (
    "@font-face{font-family:'Overpass';font-style:normal;font-weight:300;"
    "src:url(data:font/woff2;base64," + _overpass_light + ") format('woff2')}"
    "  @font-face{font-family:'Overpass';font-style:normal;font-weight:600;"
    "src:url(data:font/woff2;base64," + _overpass_semi  + ") format('woff2')}"
    "  @font-face{font-family:'Overpass';font-style:normal;font-weight:700;"
    "src:url(data:font/woff2;base64," + _overpass_bold  + ") format('woff2')}"
)
_html_out = _html_out.replace(
    "@import url('https://fonts.googleapis.com/css2?family=IBM+Plex+Mono:wght@400;500"
    "&family=IBM+Plex+Sans:wght@300;400;500;600&display=swap');",
    _font_faces
)
_html_out = _html_out.replace("'IBM Plex Mono',monospace", "'Overpass',monospace")
_html_out = _html_out.replace("'IBM Plex Sans',sans-serif", "'Overpass',sans-serif")

with open('dashboard_ocupacion_camiones.html', 'w', encoding='utf-8') as _f:
    _f.write(_html_out)


print(f'✅ Dashboard generado: dashboard_ocupacion_camiones.html')
print(f'   {len(_D["det_rows"])} registros de detalle | {len(_D["sup_rows"])} PNs | {len(_D["prov_stats"])} proveedores')


✅ Dashboard generado: dashboard_ocupacion_camiones.html
   36 registros de detalle | 10 PNs | 20 proveedores


In [240]:
with pd.ExcelWriter('resultado_optimizacion.xlsx', engine='openpyxl') as writer:
    df_detalle.to_excel(writer, sheet_name='Detalle por PN y Camion',    index=False)
    # Hoja adicional con nombre proveedor e ID embalaje
    df_detalle.to_excel(writer, sheet_name='Detalle con ID Embalaje',     index=False)
    df_resumen.to_excel(writer, sheet_name='Resumen por Camion',          index=False)
    df_resumen_pn.to_excel(writer, sheet_name='Resumen por Part Number',  index=False)
    df_sup_minima.to_excel(writer, sheet_name='Superficie minima por PN', index=False)
    df_matriz.to_excel(writer, sheet_name='Matriz Distancias Proveedores', index=True)

    # Hoja 6: Estadísticas de ocupación (imagen embebida)
    ws_stats = writer.book.create_sheet('Estadisticas de Ocupacion')
    from openpyxl.drawing.image import Image as XLImage
    from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
    from openpyxl.utils import get_column_letter
    _img = XLImage(_buf)
    _img.anchor = 'B2'
    ws_stats.add_image(_img)

    # ── Comentarios: análisis bottom 50% ────────────────────────
    _H_TITULO   = Font(name='Arial', bold=True, size=11, color='FFFFFF')
    _F_TITULO   = PatternFill('solid', start_color='1F4E79')
    _H_SUBTIT   = Font(name='Arial', bold=True, size=10, color='FFFFFF')
    _F_CAUSA    = PatternFill('solid', start_color='FCE4D6')
    _F_RECOM    = PatternFill('solid', start_color='E2EFDA')
    _F_CAM      = PatternFill('solid', start_color='BDD7EE')
    _WRAP       = Alignment(wrap_text=True, vertical='top')
    _THIN       = Side(style='thin', color='BFBFBF')
    _BORDER     = Border(left=_THIN, right=_THIN, top=_THIN, bottom=_THIN)
    _img_rows   = 22  # filas que ocupa la imagen (aprox)
    _row_ini    = _img_rows + 4

    # Título del bloque de análisis
    _umbral_txt = f'{_umbral_pct:.1f}'
    ws_stats.merge_cells(f'B{_row_ini}:H{_row_ini}')
    _tc = ws_stats[f'B{_row_ini}']
    _tc.value = f'ANÁLISIS DE CAUSAS — TOP 50% DE CAMIONES CON MENOR OCUPACIÓN VOLUMÉTRICA (≤ {_umbral_txt}%)'
    _tc.font = _H_TITULO; _tc.fill = _F_TITULO; _tc.alignment = _WRAP; _tc.border = _BORDER
    ws_stats.row_dimensions[_row_ini].height = 20

    # Encabezados de tabla
    _row_ini += 1
    _headers = ['Camión N°', 'Ocup. Vol. %', 'Ocup. Sup. %', 'Proveedores (DUNS)', 'Causa', 'Recomendación']
    _cols    = ['B', 'C', 'D', 'E', 'F', 'H']
    _widths  = [12, 14, 14, 30, 55, 55]
    for col_l, hdr, w in zip(_cols, _headers, _widths):
        _c = ws_stats[f'{col_l}{_row_ini}']
        _c.value = hdr
        _c.font  = Font(name='Arial', bold=True, size=9, color='FFFFFF')
        _c.fill  = PatternFill('solid', start_color='2E75B6')
        _c.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
        _c.border = _BORDER
        ws_stats.column_dimensions[col_l].width = w
    ws_stats.row_dimensions[_row_ini].height = 25

    # Filas de análisis
    _row_ini += 1
    for _ar in _analisis_rows:
        _max_items = max(len(_ar['causas']), len(_ar['recomendaciones']), 1)
        for _idx in range(_max_items):
            _r = _row_ini
            if _idx == 0:
                ws_stats[f'B{_r}'].value = _ar['cam_id']
                ws_stats[f'C{_r}'].value = _ar['pct_vol']
                ws_stats[f'D{_r}'].value = _ar['pct_sup']
                ws_stats[f'E{_r}'].value = _ar['proveedores']
                for _col in ['B','C','D','E']:
                    ws_stats[f'{_col}{_r}'].fill   = _F_CAM
                    ws_stats[f'{_col}{_r}'].font   = Font(name='Arial', size=9)
                    ws_stats[f'{_col}{_r}'].alignment = _WRAP
                    ws_stats[f'{_col}{_r}'].border = _BORDER
            causa = _ar['causas'][_idx] if _idx < len(_ar['causas']) else ''
            recom = _ar['recomendaciones'][_idx] if _idx < len(_ar['recomendaciones']) else ''
            ws_stats[f'F{_r}'].value = causa
            ws_stats[f'H{_r}'].value = recom
            ws_stats[f'F{_r}'].fill = _F_CAUSA; ws_stats[f'F{_r}'].font = Font(name='Arial', size=9)
            ws_stats[f'F{_r}'].alignment = _WRAP; ws_stats[f'F{_r}'].border = _BORDER
            ws_stats[f'H{_r}'].fill = _F_RECOM; ws_stats[f'H{_r}'].font = Font(name='Arial', size=9)
            ws_stats[f'H{_r}'].alignment = _WRAP; ws_stats[f'H{_r}'].border = _BORDER
            ws_stats.row_dimensions[_r].height = 60
            _row_ini += 1

    # ── Hojas de input ──────────────────────────────────────────
    df_embalajes.to_excel(writer,   sheet_name='tabla_embalajes_input',   index=False)
    df_demanda.to_excel(writer,     sheet_name='tabla_demanda_input',      index=False)
    df_apilamiento.to_excel(writer, sheet_name='tabla_apilamiento_input',  index=False)
    df_camion.to_excel(writer,      sheet_name='tabla_camion_input',       index=False)
    df_proveedores.to_excel(writer, sheet_name='tabla_proveedores_input',  index=False)

    # ── Hoja 12: Cronograma de colectas ────────────────────────
    df_cronograma.to_excel(writer, sheet_name='Cronograma de Colectas', index=False)

    # ── Colorear hojas: gris para inputs, verde para outputs ────
    _COLOR_INPUT  = '808080'  # gris
    _COLOR_OUTPUT = '70AD47'  # verde
    _hojas_input  = ['tabla_embalajes_input', 'tabla_demanda_input',
                     'tabla_apilamiento_input', 'tabla_camion_input',
                     'tabla_proveedores_input']
    _hojas_output = ['Detalle por PN y Camion', 'Detalle con ID Embalaje',
                     'Resumen por Camion', 'Resumen por Part Number',
                     'Superficie minima por PN', 'Matriz Distancias Proveedores',
                     'Estadisticas de Ocupacion', 'Cronograma de Colectas']
    for _nombre in _hojas_input:
        writer.book[_nombre].sheet_properties.tabColor = _COLOR_INPUT
    for _nombre in _hojas_output:
        writer.book[_nombre].sheet_properties.tabColor = _COLOR_OUTPUT

print('✅ Exportado correctamente a: resultado_optimizacion.xlsx')
print(f'   Hoja 1 — Detalle por PN y Camion: {len(df_detalle)} filas')
print(f'   Hoja 2 — Detalle con ID Embalaje: {len(df_detalle)} filas')
print(f'   Hoja 2 — Resumen por Camion: {len(df_resumen)} filas')
print(f'   Hoja 3 — Resumen por Part Number: {len(df_resumen_pn)} filas')
print(f'   Hoja 4 — Superficie minima por PN: {len(df_sup_minima)} filas')
print(f'   Hoja 5 — Matriz Distancias Proveedores: {len(df_proveedores)}×{len(df_proveedores)}')
print(f'   Hoja 6 — Estadísticas de Ocupación: histogramas embebidos')
print(f'   Hoja 12 — Cronograma de Colectas: {len(df_cronograma)} colectas')
print(f'   Hoja 7 — tabla_embalajes_input: {len(df_embalajes)} filas')
print(f'   Hoja 8 — tabla_demanda_input: {len(df_demanda)} filas')
print(f'   Hoja 9 — tabla_apilamiento_input: {len(df_apilamiento)} filas')
print(f'   Hoja 10 — tabla_camion_input: {len(df_camion)} fila')
print(f'   Hoja 11 — tabla_proveedores_input: {len(df_proveedores)} filas')


✅ Exportado correctamente a: resultado_optimizacion.xlsx
   Hoja 1 — Detalle por PN y Camion: 36 filas
   Hoja 2 — Detalle con ID Embalaje: 36 filas
   Hoja 2 — Resumen por Camion: 21 filas
   Hoja 3 — Resumen por Part Number: 10 filas
   Hoja 4 — Superficie minima por PN: 10 filas
   Hoja 5 — Matriz Distancias Proveedores: 20×20
   Hoja 6 — Estadísticas de Ocupación: histogramas embebidos
   Hoja 12 — Cronograma de Colectas: 22 colectas
   Hoja 7 — tabla_embalajes_input: 102 filas
   Hoja 8 — tabla_demanda_input: 100 filas
   Hoja 9 — tabla_apilamiento_input: 29 filas
   Hoja 10 — tabla_camion_input: 1 fila
   Hoja 11 — tabla_proveedores_input: 20 filas
